# 🔧 Исправление Canonical формата Balance Sheet на основе официальной отчетности

**Цель**: Создать правильный canonical формат для истории BS, который соответствует реальным данным из официальной отчетности

**Задача**: 
1. Загрузить данные из официального Balance Sheet 2024
2. Сравнить с RAW данными из БД (как данные загрузились в БД до маппинга)
3. Сравнить с CANONICAL данными из БД (как данные стали после маппинга в DataMart)
4. Выявить расхождения и понять, правильно ли препроцессинг собрал баланс
5. Создать исправленный canonical формат для 2024 года на основе официальных данных
6. Обновить данные в БД или файл истории

**Примечание**: CSV файлы истории используются DataMart как fallback, но для сравнения мы работаем напрямую с БД (RAW и CANONICAL форматы).

**Важно**: 
- Используйте данные в том же порядке, как в официальном отчете
- Проверьте порядок цифр (могут быть ошибки в масштабе)
- Убедитесь, что все статьи правильно сопоставлены с canonical метриками

**⭐ Новые метрики breakdown (опциональные)**:
- **AR**: `accounts_receivable_gross`, `allowance_for_doubtful_accounts` → `accounts_receivable_net`
- **Inventory**: `inventory_raw_materials`, `inventory_wip`, `inventory_finished_goods` → `inventory_total`
- **PPE**: `ppe_gross`, `ppe_accumulated_depreciation` → `ppe_net`
- **ROU Assets**: `rou_finance_asset`, `rou_operating_asset` → `rou_asset_total`
- **Intangibles**: `intangibles_finite_life`, `intangibles_indefinite_life` → `intangibles_total`
- **LT Debt**: `lt_debt_bonds`, `lt_debt_bank`, `lt_debt_other` → `lt_debt`
- **Lease Liabilities**: `current_finance_lease_liability`, `current_operating_lease_liability`, `noncurrent_finance_lease_liability`, `noncurrent_operating_lease_liability`
- **Другие**: `restricted_cash`, `prepaid_expenses`, `accrued_compensation`, `taxes_payable`, `interest_payable`, `current_portion_of_lt_debt`, `goodwill`, `longterm_investments`, `deferred_tax_liabilities`

Эти метрики опциональны и используются для более детального моделирования, если доступны в исходных данных.

## 🔧 Импорты и настройка

In [1]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import sys
from IPython.display import display, Markdown, HTML
from typing import Optional
from difflib import SequenceMatcher
import re

# Настройка для графиков в Jupyter
%matplotlib inline

# Определение корня проекта (ПЕРЕД импортом engine)
current_dir = Path.cwd()
if (current_dir / 'engine').exists():
    ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    ROOT = current_dir.parent.parent
else:
    nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
    ROOT = nb_path.parent.parent.parent

print(f"ROOT: {ROOT}")
print(f"Engine exists: {(ROOT / 'engine').exists()}")

# Добавляем ROOT в sys.path ПЕРЕД импортом engine
sys.path.insert(0, str(ROOT))

# Теперь импортируем engine
from engine.database.data_mart import get_data_mart

# Определяем COMPANY автоматически из пути ноутбука
COMPANY = 'us_steel'  # fallback по умолчанию
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]

croot = ROOT / 'companies' / COMPANY
print(f"Company root: {croot}")

MODEL_VERSION: Optional[str] = None

ROOT: /home
Engine exists: True
Company root: /home/companies/us_steel


## 📋 ШАГ 1: Данные из официального Balance Sheet 2024

In [2]:
# === Вставьте данные из официального Balance Sheet 2024 ===
# Данные из официальной отчетности US Steel (в том же порядке, как в отчете)

official_reporting_2024 = {
    # === ASSETS ===
    # Current Assets:
    'Cash and cash equivalents': 1367.0,  # миллионов USD
    'Receivables, less allowance': 1236.0,
    'Receivables from related parties': 162.0,
    'Inventories': 2168.0,
    'Other current assets': 299.0,
    'Total current assets': 5232.0,
    
    # Non-Current Assets:
    'Long-term restricted cash': 35.0,
    'Investments and long-term receivables': 757.0,
    'Operating lease assets': 72.0,
    'Property, plant and equipment, net': 11973.0,
    'Intangibles, net': 416.0,
    'Deferred income tax benefits': 0.0,  # указано как "—" (ноль или отсутствует)
    'Goodwill': 920.0,
    'Other noncurrent assets': 830.0,
    'Total assets': 20235.0,
    
    # === LIABILITIES ===
    # Current Liabilities:
    'Accounts payable and other accrued liabilities': 2601.0,
    'Accounts payable to related parties': 146.0,
    'Payroll and benefits payable': 295.0,
    'Accrued taxes': 131.0,
    'Accrued interest': 70.0,
    'Current operating lease liabilities': 35.0,
    'Short-term debt and current maturities of long-term debt': 95.0,
    'Total current liabilities': 3373.0,
    
    # Non-Current Liabilities:
    'Noncurrent operating lease liabilities': 44.0,
    'Long-term debt, less unamortized discount': 4078.0,
    'Employee benefits': 117.0,
    'Deferred income tax liabilities': 657.0,
    'Deferred credits and other noncurrent liabilities': 526.0,
    'Total liabilities': 8795.0,
    
    # === STOCKHOLDERS' EQUITY ===
    'Common stock issued': 288.0,
    'Treasury stock, at cost': -1446.0,  # отрицательное значение
    'Additional paid-in capital': 5307.0,
    'Retained earnings': 7219.0,
    'Accumulated other comprehensive (loss) income': -21.0,
    'Total United States Steel Corporation stockholders equity': 11347.0,
    'Noncontrolling interests': 93.0,
    'Total liabilities and stockholders equity': 20235.0
}

# Конвертируем из миллионов в доллары
official_reporting_2024 = {k: v * 1_000_000 for k, v in official_reporting_2024.items()}

print(f"✅ Загружено {len(official_reporting_2024)} статей из официального Balance Sheet 2024")

# Создаём таблицу для отображения
official_df = pd.DataFrame([
    {
        '№': i + 1,
        'Статья (официальный отчет)': item,
        'Значение (млн USD)': value / 1_000_000,
        'Значение (USD)': value
    }
    for i, (item, value) in enumerate(official_reporting_2024.items())
])

display(Markdown("### 📊 Таблица данных из официального Balance Sheet 2024"))
display(official_df.style.format({
    'Значение (млн USD)': '${:,.2f}',
    'Значение (USD)': '${:,.0f}'
}).set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
]).set_properties(**{'font-size': '9px'}))

print(f"\n📋 Всего статей: {len(official_reporting_2024)}")

✅ Загружено 37 статей из официального Balance Sheet 2024


### 📊 Таблица данных из официального Balance Sheet 2024

,№,Статья (официальный отчет),Значение (млн USD),Значение (USD)
0,1,Cash and cash equivalents,"$1,367.00","$1,367,000,000"
1,2,"Receivables, less allowance","$1,236.00","$1,236,000,000"
2,3,Receivables from related parties,$162.00,"$162,000,000"
3,4,Inventories,"$2,168.00","$2,168,000,000"
4,5,Other current assets,$299.00,"$299,000,000"
5,6,Total current assets,"$5,232.00","$5,232,000,000"
6,7,Long-term restricted cash,$35.00,"$35,000,000"
7,8,Investments and long-term receivables,$757.00,"$757,000,000"
8,9,Operating lease assets,$72.00,"$72,000,000"
9,10,"Property, plant and equipment, net","$11,973.00","$11,973,000,000"



📋 Всего статей: 37


## 📄 ШАГ 2: Загрузка данных из файла истории (опционально)

**Примечание**: Этот шаг опционален. CSV файл (`bs_history_us_steel.csv`) - это результат конвертации Excel файла через `excel_loader.py`. 

**Процесс загрузки данных**:
1. **Excel файл** → `excel_loader.py` → **CSV файл** (без маппинга, как есть)
2. **CSV файл** → `load_history_csv_to_db` → **RAW БД** (загружается как есть)
3. **RAW БД** → `normalize_to_canonical` + `excel_loader.yaml` → **CANONICAL БД** (применяется маппинг)

CSV файлы используются DataMart как fallback, но основная работа идет с данными из БД (ШАГ 3). 

Если хотите проверить, что данные из Excel правильно конвертировались в CSV и загрузились в БД, можете использовать этот шаг. Иначе переходите к ШАГ 3.

In [3]:
# === Загрузка данных из файла истории BS ===

hist_file = croot / "history" / "bs_history_us_steel.csv"

if not hist_file.exists():
    print(f"❌ Файл истории не найден: {hist_file}")
    hist_2024_raw = {}
else:
    hist_df = pd.read_csv(hist_file)
    print(f"✅ Файл истории загружен: {hist_file.name}")
    print(f"   Структура: {hist_df.shape[0]} строк, {hist_df.shape[1]} колонок")
    
    # Получаем данные 2024 из файла истории
    hist_2024_raw = {}
    if '2024' in hist_df.columns:
        for _, row in hist_df.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['2024'], errors='coerce')
            if pd.notna(value):
                hist_2024_raw[metric] = float(value)
        
        print(f"\n✅ Загружено {len(hist_2024_raw)} статей из файла истории для 2024 года")
        
        # Создаём таблицу для отображения
        hist_df_display = pd.DataFrame([
            {
                '№': i + 1,
                'Статья (файл истории)': item,
                'Значение (млн USD)': value / 1_000_000,
                'Значение (USD)': value
            }
            for i, (item, value) in enumerate(sorted(hist_2024_raw.items()))
        ])
        
        display(Markdown("### 📄 Таблица данных из файла истории (2024 год)"))
        display(hist_df_display.style.format({
        'Значение (млн USD)': '${:,.2f}',
        'Значение (USD)': '${:,.0f}'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '9px'}))
    else:
        print(f"❌ Колонка '2024' не найдена в файле истории")
        print(f"   Доступные колонки: {list(hist_df.columns)}")

✅ Файл истории загружен: bs_history_us_steel.csv
   Структура: 25 строк, 16 колонок

✅ Загружено 25 статей из файла истории для 2024 года


### 📄 Таблица данных из файла истории (2024 год)

,№,Статья (файл истории),Значение (млн USD),Значение (USD)
0,1,amortization,$20.00,"$20,000,000"
1,2,cogs,"$14,060.00","$14,060,000,000"
2,3,depreciation_finance_leases,$56.00,"$56,000,000"
3,4,depreciation_owned,"$-14,084.00","$-14,084,000,000"
4,5,depreciation_rou,$14.00,"$14,000,000"
5,6,ebit,$240.00,"$240,000,000"
6,7,ebitda,$422.00,"$422,000,000"
7,8,ebt,$440.00,"$440,000,000"
8,9,eps_basic,$0.00,$2
9,10,eps_diluted,$0.00,$2


## 🗄️ ШАГ 3: Загрузка данных из БД (canonical формат)

In [4]:
# === Загрузка данных из БД: RAW (до маппинга) и CANONICAL (после маппинга) ===

mart = get_data_mart(ROOT, COMPANY)

# RAW данные - как они загрузились в БД из файла истории (БЕЗ маппинга)
raw_bs = mart.get_history_balance_sheet(canonical=False)
print("📥 RAW данные из БД (до маппинга):")
print(f"   Загружено: {raw_bs.shape[0]} строк, {raw_bs.shape[1]} колонок")

# CANONICAL данные - после применения маппинга в DataMart
canonical_bs = mart.get_history_balance_sheet(canonical=True)
print(f"\n📤 CANONICAL данные из БД (после маппинга):")
print(f"   Загружено: {canonical_bs.shape[0]} строк, {canonical_bs.shape[1]} колонок")

mart.close()

# Получаем данные 2024 из RAW формата (до маппинга)
raw_2024 = {}
if not raw_bs.empty:
    if 'year' in raw_bs.columns and 'metric' in raw_bs.columns:
        # Long format
        bs_2024_raw = raw_bs[raw_bs['year'] == 2024]
        for _, row in bs_2024_raw.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['value'], errors='coerce')
            if pd.notna(value):
                raw_2024[metric] = float(value)
    elif 'metric' in raw_bs.columns:
        # Wide format
        if '2024' in raw_bs.columns:
            for _, row in raw_bs.iterrows():
                metric = row['metric']
                value = pd.to_numeric(row['2024'], errors='coerce')
                if pd.notna(value):
                    raw_2024[metric] = float(value)
    
    print(f"\n✅ Загружено {len(raw_2024)} статей из RAW формата для 2024 года")
    
    # Создаём таблицу для отображения RAW данных
    raw_df_display = pd.DataFrame([
        {
            '№': i + 1,
            'Статья (RAW - из файла истории)': item,
            'Значение (млн USD)': value / 1_000_000,
            'Значение (USD)': value
        }
        for i, (item, value) in enumerate(sorted(raw_2024.items()))
    ])
    
    display(Markdown("### 📥 Таблица RAW данных из БД (2024 год, до маппинга)"))
    display(raw_df_display.style.format({
        'Значение (млн USD)': '${:,.2f}',
        'Значение (USD)': '${:,.0f}'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '9px'}))
else:
    print("⚠️ RAW BS пуст или не найден в БД")

# Получаем данные 2024 из canonical формата (после маппинга)
canonical_2024 = {}
if not canonical_bs.empty:
    print(f"\n✅ Canonical BS загружен из БД: {canonical_bs.shape[0]} записей")
    
    if 'year' in canonical_bs.columns and 'metric' in canonical_bs.columns:
        # Long format
        bs_2024 = canonical_bs[canonical_bs['year'] == 2024]
        for _, row in bs_2024.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['value'], errors='coerce')
            if pd.notna(value):
                canonical_2024[metric] = float(value)
    elif 'metric' in canonical_bs.columns:
        # Wide format
        if '2024' in canonical_bs.columns:
            for _, row in canonical_bs.iterrows():
                metric = row['metric']
                value = pd.to_numeric(row['2024'], errors='coerce')
                if pd.notna(value):
                    canonical_2024[metric] = float(value)
    
    print(f"\n✅ Загружено {len(canonical_2024)} статей из canonical формата для 2024 года")
    
    # Создаём таблицу для отображения canonical данных
    canonical_df_display = pd.DataFrame([
        {
            '№': i + 1,
            'Статья (canonical - после маппинга)': item,
            'Значение (млн USD)': value / 1_000_000,
            'Значение (USD)': value
        }
        for i, (item, value) in enumerate(sorted(canonical_2024.items()))
    ])
    
    display(Markdown("### 📤 Таблица CANONICAL данных из БД (2024 год, после маппинга)"))
    display(canonical_df_display.style.format({
        'Значение (млн USD)': '${:,.2f}',
        'Значение (USD)': '${:,.0f}'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '9px'}))
    
    # Сравнение RAW vs CANONICAL
    print("\n" + "="*80)
    print("🔍 Сравнение RAW vs CANONICAL:")
    print("="*80)
    
    comparison_raw_canonical = []
    all_metrics = set(raw_2024.keys()) | set(canonical_2024.keys())
    
    for metric in sorted(all_metrics):
        raw_val = raw_2024.get(metric)
        canonical_val = canonical_2024.get(metric)
        
        if raw_val is not None and canonical_val is not None:
            diff = abs(raw_val - canonical_val)
            if diff > 1000:  # Разница больше $1K
                comparison_raw_canonical.append({
                    'Метрика': metric,
                    'RAW значение': f"${raw_val:,.0f}",
                    'CANONICAL значение': f"${canonical_val:,.0f}",
                    'Разница': f"${diff:,.0f}",
                    'Статус': '⚠️ Изменено'
                })
            else:
                comparison_raw_canonical.append({
                    'Метрика': metric,
                    'RAW значение': f"${raw_val:,.0f}",
                    'CANONICAL значение': f"${canonical_val:,.0f}",
                    'Разница': f"${diff:,.0f}",
                    'Статус': '✅ Совпадает'
                })
        elif raw_val is not None:
            comparison_raw_canonical.append({
                'Метрика': metric,
                'RAW значение': f"${raw_val:,.0f}",
                'CANONICAL значение': '-',
                'Разница': '-',
                'Статус': '⚠️ Только в RAW'
            })
        elif canonical_val is not None:
            comparison_raw_canonical.append({
                'Метрика': metric,
                'RAW значение': '-',
                'CANONICAL значение': f"${canonical_val:,.0f}",
                'Разница': '-',
                'Статус': 'ℹ️ Добавлено в canonical'
            })
    
    if comparison_raw_canonical:
        comparison_df_raw_canon = pd.DataFrame(comparison_raw_canonical)
        display(Markdown("#### 📊 Сравнение RAW vs CANONICAL:"))
        display(comparison_df_raw_canon.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '9px'}))
        
        changed = len([x for x in comparison_raw_canonical if 'Изменено' in x['Статус']])
        added = len([x for x in comparison_raw_canonical if 'Добавлено' in x['Статус']])
        print(f"\n📋 Сводка:")
        print(f"  Метрик изменено при маппинге: {changed}")
        print(f"  Метрик добавлено в canonical: {added}")
        print(f"  Метрик совпадает: {len(comparison_raw_canonical) - changed - added}")
else:
    print("⚠️ Canonical BS пуст или не найден в БД")

📥 RAW данные из БД (до маппинга):
   Загружено: 76 строк, 16 колонок

📤 CANONICAL данные из БД (после маппинга):
   Загружено: 51 строк, 16 колонок

✅ Загружено 50 статей из RAW формата для 2024 года


### 📥 Таблица RAW данных из БД (2024 год, до маппинга)

,№,Статья (RAW - из файла истории),Значение (млн USD),Значение (USD)
0,1,accounts_payable,"$2,601.00","$2,601,000,000"
1,2,accounts_payable_related_parties,$146.00,"$146,000,000"
2,3,accounts_receivable,"$1,236.00","$1,236,000,000"
3,4,accrued_interest,$70.00,"$70,000,000"
4,5,accrued_liabilities,$146.00,"$146,000,000"
5,6,accrued_taxes,$131.00,"$131,000,000"
6,7,aoci,$-21.00,"$-21,000,000"
7,8,ap_related_parties,$0.00,$0
8,9,apic,"$5,307.00","$5,307,000,000"
9,10,cash,"$1,367.00","$1,367,000,000"



✅ Canonical BS загружен из БД: 51 записей

✅ Загружено 51 статей из canonical формата для 2024 года


### 📤 Таблица CANONICAL данных из БД (2024 год, после маппинга)

,№,Статья (canonical - после маппинга),Значение (млн USD),Значение (USD)
0,1,accounts_payable,"$2,601.00","$2,601,000,000"
1,2,accounts_payable_related_parties,$146.00,"$146,000,000"
2,3,accounts_receivable,"$1,236.00","$1,236,000,000"
3,4,accrued_interest,$70.00,"$70,000,000"
4,5,accrued_liabilities,$146.00,"$146,000,000"
5,6,accrued_taxes,$131.00,"$131,000,000"
6,7,aoci,$-21.00,"$-21,000,000"
7,8,ap_related_parties,$0.00,$0
8,9,apic,"$5,307.00","$5,307,000,000"
9,10,cash,"$1,367.00","$1,367,000,000"



🔍 Сравнение RAW vs CANONICAL:


#### 📊 Сравнение RAW vs CANONICAL:

,Метрика,RAW значение,CANONICAL значение,Разница,Статус
0,accounts_payable,"$2,601,000,000","$2,601,000,000",$0,✅ Совпадает
1,accounts_payable_related_parties,"$146,000,000","$146,000,000",$0,✅ Совпадает
2,accounts_receivable,"$1,236,000,000","$1,236,000,000",$0,✅ Совпадает
3,accrued_interest,"$70,000,000","$70,000,000",$0,✅ Совпадает
4,accrued_liabilities,"$146,000,000","$146,000,000",$0,✅ Совпадает
5,accrued_taxes,"$131,000,000","$131,000,000",$0,✅ Совпадает
6,aoci,"$-21,000,000","$-21,000,000",$0,✅ Совпадает
7,ap_related_parties,$0,$0,$0,✅ Совпадает
8,apic,"$5,307,000,000","$5,307,000,000",$0,✅ Совпадает
9,cash,"$1,367,000,000","$1,367,000,000",$0,✅ Совпадает



📋 Сводка:
  Метрик изменено при маппинге: 0
  Метрик добавлено в canonical: 1
  Метрик совпадает: 50


## 🔗 ШАГ 4: Загрузка маппинга статей

**Примечание**: Маппинг статей из официального отчета на canonical метрики происходит автоматически в препроцессинге при загрузке данных в БД через DataMart. 

Здесь мы загружаем конфигурацию маппинга из `excel_loader.yaml` только для **понимания**, какие статьи из официального отчета соответствуют каким canonical метрикам. Это нужно для:
- Сопоставления данных из официального отчета с canonical метриками
- Понимания, какие статьи были объединены или переименованы
- Проверки правильности маппинга

**Важно**: Если данные уже загружены в БД, маппинг уже применён. Наша задача - убедиться, что значения в файле истории соответствуют официальной отчетности.

In [5]:
# === Загрузка конфигурации маппинга из excel_loader.yaml ===
# Примечание: Маппинг для BS находится в excel_loader.yaml (не в отдельном bs_mapping.yaml)
# Это файл для загрузки данных из Excel, который содержит маппинг метрик

excel_loader_path = croot / "configs" / "excel_loader.yaml"
mapping_config = {}
required_metrics = {}

if excel_loader_path.exists():
    print(f"✅ Файл найден: {excel_loader_path.relative_to(ROOT)}")
    with open(excel_loader_path, 'r', encoding='utf-8') as f:
        mapping_config = yaml.safe_load(f)
    
    # Проверяем структуру файла
    print(f"\n📋 Доступные секции в excel_loader.yaml: {list(mapping_config.keys())}")
    
    # Ищем секцию BS
    bs_mapping = mapping_config.get('history', {}).get('BS', {})
    if not bs_mapping:
        # Пробуем напрямую
        bs_mapping = mapping_config.get('BS', {})
    
    if bs_mapping:
        print("✅ Найдена конфигурация маппинга для BS")
        required_metrics = bs_mapping.get('required_metrics', {})
        
        if required_metrics:
            print(f"\n📊 Найдено {len(required_metrics)} метрик в маппинге")
            print(f"\nТребуемые метрики (с алиасами):")
            
            metrics_with_aliases = []
            metrics_without_aliases = []
            
            for metric, config in required_metrics.items():
                if isinstance(config, dict):
                    aliases = config.get('aliases', [])
                    if aliases:
                        metrics_with_aliases.append((metric, aliases))
                    else:
                        metrics_without_aliases.append(metric)
                else:
                    metrics_without_aliases.append(metric)
            
            # Показываем метрики с алиасами
            if metrics_with_aliases:
                print("\n  Метрики с алиасами:")
                for metric, aliases in metrics_with_aliases[:15]:
                    print(f"    {metric}: {', '.join(aliases[:3])}{'...' if len(aliases) > 3 else ''}")
            
            # Показываем метрики без алиасов
            if metrics_without_aliases:
                print(f"\n  Метрики без алиасов ({len(metrics_without_aliases)}): {', '.join(metrics_without_aliases[:10])}{'...' if len(metrics_without_aliases) > 10 else ''}")
        else:
            print("⚠️ Секция 'required_metrics' не найдена в конфигурации BS")
    else:
        print("⚠️ Конфигурация BS не найдена в excel_loader.yaml")
        print(f"   Доступные секции: {list(mapping_config.keys())}")
        if 'history' in mapping_config:
            print(f"   Секции в 'history': {list(mapping_config['history'].keys())}")
else:
    print(f"⚠️ Файл excel_loader.yaml не найден: {excel_loader_path}")
    print(f"   Проверьте путь: {croot / 'configs'}")

print(f"\n✅ Маппинг загружен: {len(required_metrics)} метрик")

✅ Файл найден: companies/us_steel/configs/excel_loader.yaml

📋 Доступные секции в excel_loader.yaml: ['meta', 'history', 'schedules', 'notes', 'segments', 'operational_drivers', 'macro', 'validation', 'loader_defaults']
✅ Найдена конфигурация маппинга для BS

📊 Найдено 51 метрик в маппинге

Требуемые метрики (с алиасами):

  Метрики с алиасами:
    cash: cash_and_equivalents
    accounts_receivable: accounts_receivable_net, ar, receivables_less_allowance
    accounts_receivable_gross: ar_gross
    allowance_for_doubtful_accounts: allowance_for_bad_debts
    receivables_related_parties: receivables_from_related_parties
    inventory: inventory_total, inv
    inventory_raw_materials: raw_materials
    inventory_wip: work_in_progress, wip
    inventory_finished_goods: finished_goods
    prepaid_expenses: prepaids
    ppe_gross: property_plant_and_equipment_gross, ppe_gross_cost
    ppe_accumulated_depreciation: ppe_accum_dep, accumulated_depreciation_ppe
    ppe_net: property_plant_and_eq

## 🔍 ШАГ 5: Детальное сравнение всех источников

**Цель**: Сравнить данные из официального отчета с данными в БД на разных этапах обработки:
1. **Официальный отчет** → исходные данные из отчетности
2. **RAW данные из БД** → как данные загрузились в БД из CSV (до маппинга)
3. **CANONICAL данные из БД** → как данные стали после маппинга в DataMart (препроцессинг)

**Опционально**: Файл истории (CSV) - для проверки, что CSV правильно загрузился в БД

Это позволит понять:
- Правильно ли данные из официального отчета попали в БД (RAW)
- Правильно ли препроцессинг применил маппинг (RAW → CANONICAL)
- Где происходят расхождения и что нужно исправить

In [6]:
# === Детальное сравнение всех источников с учетом маппинга ===

def similarity(a, b):
    """Вычисляет схожесть двух строк (0-1)"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def normalize_name(name):
    """Нормализует название для сравнения"""
    normalized = re.sub(r'[^a-z0-9\s]', '', name.lower())
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

def find_best_match(official_name, candidate_list, threshold=0.7):
    """Находит лучшее совпадение для названия из официального отчета"""
    best_match = None
    best_score = 0.0
    
    normalized_official = normalize_name(official_name)
    
    for candidate in candidate_list:
        normalized_candidate = normalize_name(candidate)
        score = similarity(normalized_official, normalized_candidate)
        
        if normalized_official == normalized_candidate:
            return candidate, 1.0
        
        if score > best_score:
            best_score = score
            best_match = candidate
    
    if best_score >= threshold:
        return best_match, best_score
    return None, best_score

display(Markdown("### 🔍 Детальное сравнение данных 2024"))

if not official_reporting_2024:
    print("❌ Данные из официальной отчетности не загружены. Запустите ячейку ШАГ 1.")
elif 'raw_2024' not in globals() or not raw_2024:
    print("❌ RAW данные из БД не загружены. Запустите ячейку ШАГ 3.")
elif not canonical_2024:
    print("❌ Данные из canonical формата не загружены. Запустите ячейку ШАГ 3.")
else:
    print(f"✅ Основные источники данных загружены:")
    print(f"  1. Официальный отчет: {len(official_reporting_2024)} статей")
    print(f"  2. RAW из БД (до маппинга): {len(raw_2024)} статей")
    print(f"  3. CANONICAL из БД (после маппинга): {len(canonical_2024)} статей")
    
    # Опционально: файл истории
    if 'hist_2024_raw' in globals() and hist_2024_raw:
        print(f"  4. Файл истории (опционально): {len(hist_2024_raw)} статей")
    else:
        print(f"  4. Файл истории: не загружен (опционально)")
    
    # Создаём маппинг алиасов (alias -> canonical)
    aliases_map = {}
    # Обратный маппинг (canonical -> aliases) для поиска
    canonical_to_aliases = {}
    
    for metric, config in required_metrics.items():
        aliases = config.get('aliases', [])
        canonical_to_aliases[metric] = aliases
        for alias in aliases:
            aliases_map[alias.lower()] = metric
        aliases_map[metric.lower()] = metric
    
    # Создаём маппинг официальных названий на canonical метрики
    # На основе типичных названий из отчетности и маппинга из YAML
    official_to_canonical_map = {
        'Accounts payable and other accrued liabilities': 'accounts_payable',
        'Accounts payable to related parties': 'accounts_payable_related_parties',
        'Current operating lease liabilities': 'lease_liab_current',
        'Noncurrent operating lease liabilities': 'lease_liab_noncurrent',
        'Deferred income tax liabilities': 'dtl',
        'Deferred income tax benefits': 'dta',
        'Cash and cash equivalents': 'cash',
        'Receivables, less allowance': 'accounts_receivable',
        'Operating lease assets': 'rou_asset',
        'Property, plant and equipment, net': 'ppe_net',
        'Short-term debt and current maturities of long-term debt': 'st_debt',
        'Long-term debt, less unamortized discount': 'long_term_debt',
        'Additional paid-in capital': 'apic',
        'Accumulated other comprehensive (loss) income': 'aoci',
        'Total United States Steel Corporation stockholders equity': 'total_equity',
        'Noncontrolling interests': 'nci',
        'Total liabilities and stockholders equity': 'total_liab_equity',
        'Long-term restricted cash': 'restricted_cash',  # ВАЖНО: не cash!
    }
    
    # Создаём сравнительную таблицу
    comparison_data = []
    
    for official_item, official_val in official_reporting_2024.items():
        # Ищем совпадение в файле истории (опционально)
        # Используем тот же маппинг, что и для RAW данных
        hist_match = None
        if 'hist_2024_raw' in globals() and hist_2024_raw:
            # Сначала проверяем явный маппинг
            if official_item in official_to_canonical_map:
                canonical_name = official_to_canonical_map[official_item]
                # Ищем canonical метрику в файле истории
                if canonical_name in hist_2024_raw:
                    hist_match = canonical_name
                # Также проверяем алиасы
                if not hist_match and canonical_name in canonical_to_aliases:
                    for alias in canonical_to_aliases[canonical_name]:
                        if alias in hist_2024_raw:
                            hist_match = alias
                            break
            
            # Если не нашли через маппинг, ищем по названию
            # НО исключаем старые/неправильные метрики
            if not hist_match:
                # Исключаем старые метрики, которые могут быть перепутаны
                excluded_hist_metrics = set()
                
                # Для noncurrent lease liabilities не ищем current_lease_liability
                if 'noncurrent' in official_item.lower() and 'lease' in official_item.lower():
                    excluded_hist_metrics.add('current_lease_liability')
                    excluded_hist_metrics.add('lease_liab_current')
                
                # Для accounts_payable не ищем accounts_payable_related_parties
                if 'accounts payable and other accrued liabilities' in official_item.lower():
                    excluded_hist_metrics.add('accounts_payable_related_parties')
                
                # Для dtl не ищем other_current_liabilities
                if 'deferred income tax liabilities' in official_item.lower():
                    excluded_hist_metrics.add('other_current_liabilities')
                
                available_hist_metrics = [m for m in hist_2024_raw.keys() if m not in excluded_hist_metrics]
                hist_match, hist_score = find_best_match(official_item, available_hist_metrics, threshold=0.6)
        
        # Ищем совпадение в RAW данных из БД
        # Сначала проверяем явный маппинг
        raw_match = None
        if official_item in official_to_canonical_map:
            canonical_name = official_to_canonical_map[official_item]
            # Ищем canonical метрику в RAW данных (она может быть там под своим именем)
            if canonical_name in raw_2024:
                raw_match = canonical_name
            # Также проверяем алиасы
            if not raw_match and canonical_name in canonical_to_aliases:
                for alias in canonical_to_aliases[canonical_name]:
                    if alias in raw_2024:
                        raw_match = alias
                        break
        
        # Если не нашли через маппинг, ищем по названию
        if not raw_match:
            raw_match, raw_score = find_best_match(official_item, raw_2024.keys(), threshold=0.6)
        
        # ПРИОРИТЕТ 1: Используем явный маппинг официальных названий
        explicit_canonical_match = official_to_canonical_map.get(official_item)
        
        # ПРИОРИТЕТ 2: Ищем через алиасы (нормализуем название)
        alias_match = None
        official_normalized = normalize_name(official_item).lower()
        
        # Проверяем прямое совпадение с алиасами
        if official_normalized in aliases_map:
            alias_match = aliases_map[official_normalized]
        else:
            # Проверяем частичное совпадение с алиасами
            for alias, canonical in aliases_map.items():
                if official_normalized in alias or alias in official_normalized:
                    alias_match = canonical
                    break
        
        # ПРИОРИТЕТ 3: Проверяем алиасы canonical метрик
        alias_from_canonical = None
        if not explicit_canonical_match and not alias_match:
            for canonical_metric, aliases_list in canonical_to_aliases.items():
                for alias in aliases_list:
                    alias_normalized = normalize_name(alias).lower()
                    # Проверяем точное совпадение или что официальное название содержит алиас
                    if (official_normalized == alias_normalized or 
                        alias_normalized in official_normalized):
                        alias_from_canonical = canonical_metric
                        break
                if alias_from_canonical:
                    break
        
        # ПРИОРИТЕТ 4: Ищем совпадение по названию в canonical метриках
        # НО исключаем метрики, которые уже найдены через маппинг
        canonical_match = None
        canonical_score = 0.0
        if not explicit_canonical_match and not alias_match and not alias_from_canonical:
            # Исключаем метрики, которые могут быть перепутаны
            # Например, не ищем "cash" если ищем "restricted_cash"
            excluded_metrics = set()
            if 'restricted' in official_normalized or 'restricted' in official_item.lower():
                excluded_metrics.add('cash')  # Не путать restricted_cash с cash
            
            available_metrics = [m for m in canonical_2024.keys() if m not in excluded_metrics]
            canonical_match, canonical_score = find_best_match(official_item, available_metrics, threshold=0.6)
        
        # Используем найденное совпадение в порядке приоритета
        final_canonical_match = (explicit_canonical_match or 
                                alias_match or 
                                alias_from_canonical or 
                                canonical_match)
        
        # Проверяем, что найденная метрика существует в canonical данных
        if final_canonical_match and final_canonical_match not in canonical_2024:
            final_canonical_match = None
        
        # Получаем значения (файл истории опционален)
        hist_val = None
        if 'hist_2024_raw' in globals() and hist_2024_raw and hist_match:
            hist_val = hist_2024_raw.get(hist_match)
        
        raw_val = raw_2024.get(raw_match) if raw_match else None
        canonical_val = canonical_2024.get(final_canonical_match) if final_canonical_match else None
        
        # Проверяем расхождения
        issues = []
        
        # Сравнение с файлом истории (опционально)
        if hist_match and hist_val is not None:
            diff = abs(official_val - hist_val)
            diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
            if diff > 1000:
                issues.append(f"Офиц. vs История: ${diff:,.0f} ({diff_pct:.1f}%)")
        
        # Основное сравнение: Официальный отчет vs RAW БД
        if raw_match and raw_val is not None:
            diff = abs(official_val - raw_val)
            diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
            if diff > 1000:
                issues.append(f"Офиц. vs RAW БД: ${diff:,.0f} ({diff_pct:.1f}%)")
        
        # Основное сравнение: Официальный отчет vs CANONICAL БД
        if final_canonical_match and canonical_val is not None:
            diff = abs(official_val - canonical_val)
            diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
            if diff > 1000:
                issues.append(f"Офиц. vs Canonical: ${diff:,.0f} ({diff_pct:.1f}%)")
        
        # Проверяем расхождения между этапами обработки в БД
        if hist_val is not None and raw_val is not None:
            diff = abs(hist_val - raw_val)
            if diff > 1000:
                issues.append(f"История vs RAW БД: ${diff:,.0f}")
        
        # Ключевое сравнение: RAW vs CANONICAL (как препроцессинг изменил данные)
        if raw_val is not None and canonical_val is not None:
            diff = abs(raw_val - canonical_val)
            if diff > 1000:
                issues.append(f"RAW vs Canonical: ${diff:,.0f}")
        
        comparison_data.append({
            'Официальный отчет': official_item,
            'Значение (офиц.)': f"${official_val:,.0f}",
            'Совпадение (история)': hist_match if hist_match else "-",
            'Значение (история)': f"${hist_val:,.0f}" if hist_val is not None else "-",
            'Совпадение (RAW БД)': raw_match if raw_match else "-",
            'Значение (RAW БД)': f"${raw_val:,.0f}" if raw_val is not None else "-",
            'Совпадение (canonical)': final_canonical_match if final_canonical_match else "-",
            'Значение (canonical)': f"${canonical_val:,.0f}" if canonical_val is not None else "-",
            'Проблемы': '; '.join(issues) if issues else '✅'
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Показываем проблемные статьи
    problematic = comparison_df[comparison_df['Проблемы'] != '✅']
    if not problematic.empty:
        display(Markdown("#### ❌ Статьи с расхождениями:"))
        problematic_display = problematic[['Официальный отчет', 'Значение (офиц.)', 'Совпадение (история)', 
                             'Значение (история)', 'Совпадение (RAW БД)', 'Значение (RAW БД)',
                             'Совпадение (canonical)', 'Значение (canonical)', 'Проблемы']]
        display(problematic_display.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'table', 'props': [('font-size', '9px')]}
        ]).set_properties(**{'font-size': '9px'}))
    
    # Статьи, которые не были найдены в БД (основной критерий)
    not_found = comparison_df[(comparison_df['Совпадение (RAW БД)'] == '-') &
                               (comparison_df['Совпадение (canonical)'] == '-')]
    if not not_found.empty:
        display(Markdown("#### ⚠️ Статьи из официального отчета, не найденные в модели:"))
        not_found_display = not_found[['Официальный отчет', 'Значение (офиц.)']]
        display(not_found_display.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '9px'}))
    
    # Показываем полную таблицу с уменьшенным шрифтом
    display(Markdown("#### 📊 Полная таблица сравнения:"))
    display(comparison_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left'), ('white-space', 'nowrap')]},
        {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left'), ('white-space', 'nowrap')]},
        {'selector': 'table', 'props': [('font-size', '8px'), ('width', '100%')]}
    ]).set_properties(**{'font-size': '8px', 'max-width': '120px', 'overflow': 'hidden', 'text-overflow': 'ellipsis'}))
    
    # Сводка
    print(f"\n📋 Сводка сравнения:")
    print(f"  Всего статей в официальном отчете: {len(official_reporting_2024)}")
    
    # Подсчитываем совпадения
    hist_matches = len([x for x in comparison_df['Совпадение (история)'] if x != '-']) if 'hist_2024_raw' in globals() else 0
    raw_matches = len([x for x in comparison_df['Совпадение (RAW БД)'] if x != '-'])
    canonical_matches = len([x for x in comparison_df['Совпадение (canonical)'] if x != '-'])
    
    print(f"  Найдено совпадений в RAW БД: {raw_matches} (основной источник)")
    print(f"  Найдено совпадений в CANONICAL БД: {canonical_matches} (после маппинга)")
    if hist_matches > 0:
        print(f"  Найдено совпадений в файле истории: {hist_matches} (опционально)")
    
    print(f"  Статей с расхождениями: {len(problematic)}")
    print(f"  Статей не найденных в БД: {len(not_found)}")
    print(f"  Статей без проблем: {len(comparison_df) - len(problematic)}")
    
    # Дополнительная сводка по этапам обработки
    print(f"\n📊 Анализ этапов обработки:")
    
    # Подсчитываем изменения при загрузке в БД (история → RAW)
    hist_raw_diff = 0
    for _, row in comparison_df.iterrows():
        hist_match = row['Совпадение (история)']
        raw_match = row['Совпадение (RAW БД)']
        hist_val = row['Значение (история)']
        raw_val = row['Значение (RAW БД)']
        
        if hist_match != '-' and raw_match != '-' and hist_val != '-' and raw_val != '-':
            # Извлекаем числовые значения из строк вида "$1,234,567"
            try:
                hist_num = float(hist_val.replace('$', '').replace(',', ''))
                raw_num = float(raw_val.replace('$', '').replace(',', ''))
                if abs(hist_num - raw_num) > 1000:  # Разница больше $1K
                    hist_raw_diff += 1
            except (ValueError, AttributeError):
                pass
    
    # Подсчитываем изменения при маппинге (RAW → CANONICAL)
    raw_canonical_diff = 0
    for _, row in comparison_df.iterrows():
        raw_match = row['Совпадение (RAW БД)']
        canonical_match = row['Совпадение (canonical)']
        raw_val = row['Значение (RAW БД)']
        canonical_val = row['Значение (canonical)']
        
        if raw_match != '-' and canonical_match != '-' and raw_val != '-' and canonical_val != '-':
            try:
                raw_num = float(raw_val.replace('$', '').replace(',', ''))
                canonical_num = float(canonical_val.replace('$', '').replace(',', ''))
                if abs(raw_num - canonical_num) > 1000:  # Разница больше $1K
                    raw_canonical_diff += 1
            except (ValueError, AttributeError):
                pass
    
    print(f"  Изменений при загрузке в БД (история → RAW): {hist_raw_diff}")
    print(f"  Изменений при маппинге (RAW → CANONICAL): {raw_canonical_diff}")
    
    # Показываем примеры изменений
    if hist_raw_diff > 0:
        print(f"\n  Примеры изменений при загрузке в БД:")
        count = 0
        for _, row in comparison_df.iterrows():
            if count >= 3:
                break
            hist_match = row['Совпадение (история)']
            raw_match = row['Совпадение (RAW БД)']
            hist_val = row['Значение (история)']
            raw_val = row['Значение (RAW БД)']
            
            if hist_match != '-' and raw_match != '-' and hist_val != '-' and raw_val != '-':
                try:
                    hist_num = float(hist_val.replace('$', '').replace(',', ''))
                    raw_num = float(raw_val.replace('$', '').replace(',', ''))
                    if abs(hist_num - raw_num) > 1000:
                        print(f"    {row['Официальный отчет']}: История=${hist_num:,.0f} → RAW=${raw_num:,.0f} (diff=${abs(hist_num-raw_num):,.0f})")
                        count += 1
                except (ValueError, AttributeError):
                    pass
    
    if raw_canonical_diff > 0:
        print(f"\n  Примеры изменений при маппинге:")
        count = 0
        for _, row in comparison_df.iterrows():
            if count >= 3:
                break
            raw_match = row['Совпадение (RAW БД)']
            canonical_match = row['Совпадение (canonical)']
            raw_val = row['Значение (RAW БД)']
            canonical_val = row['Значение (canonical)']
            
            if raw_match != '-' and canonical_match != '-' and raw_val != '-' and canonical_val != '-':
                try:
                    raw_num = float(raw_val.replace('$', '').replace(',', ''))
                    canonical_num = float(canonical_val.replace('$', '').replace(',', ''))
                    if abs(raw_num - canonical_num) > 1000:
                        print(f"    {row['Официальный отчет']}: RAW=${raw_num:,.0f} → CANONICAL=${canonical_num:,.0f} (diff=${abs(raw_num-canonical_num):,.0f})")
                        count += 1
                except (ValueError, AttributeError):
                    pass

### 🔍 Детальное сравнение данных 2024

✅ Основные источники данных загружены:
  1. Официальный отчет: 37 статей
  2. RAW из БД (до маппинга): 50 статей
  3. CANONICAL из БД (после маппинга): 51 статей
  4. Файл истории (опционально): 25 статей


#### ❌ Статьи с расхождениями:

,Официальный отчет,Значение (офиц.),Совпадение (история),Значение (история),Совпадение (RAW БД),Значение (RAW БД),Совпадение (canonical),Значение (canonical),Проблемы
2,Receivables from related parties,"$162,000,000",-,-,receivables_related_parties,"$162,000,000",accounts_receivable,"$1,236,000,000","Офиц. vs Canonical: $1,074,000,000 (663.0%); RAW vs Canonical: $1,074,000,000"
4,Other current assets,"$299,000,000",other_expenses,"$15,456,000,000",other_current_assets,"$299,000,000",other_current_assets,"$299,000,000","Офиц. vs История: $15,157,000,000 (5069.2%); История vs RAW БД: $15,157,000,000"
7,Investments and long-term receivables,"$757,000,000",-,-,investments_and_long_term_receivables,"$757,000,000",inventory,"$2,168,000,000","Офиц. vs Canonical: $1,411,000,000 (186.4%); RAW vs Canonical: $1,411,000,000"
11,Deferred income tax benefits,$0,-,-,dta,"$-660,000,000",dta,"$-660,000,000","Офиц. vs RAW БД: $660,000,000 (0.0%); Офиц. vs Canonical: $660,000,000 (0.0%)"
14,Total assets,"$20,235,000,000",total_da,"$913,000,000",total_assets,"$20,235,000,000",total_assets,"$20,235,000,000","Офиц. vs История: $19,322,000,000 (95.5%); История vs RAW БД: $19,322,000,000"
18,Accrued taxes,"$131,000,000",-,-,accrued_taxes,"$131,000,000",accrued_liabilities,"$146,000,000","Офиц. vs Canonical: $15,000,000 (11.5%); RAW vs Canonical: $15,000,000"
19,Accrued interest,"$70,000,000",lease_interest,"$14,000,000",accrued_interest,"$70,000,000",accrued_liabilities,"$146,000,000","Офиц. vs История: $56,000,000 (80.0%); Офиц. vs Canonical: $76,000,000 (108.6%); История vs RAW БД: $56,000,000; RAW vs Canonical: $76,000,000"
20,Current operating lease liabilities,"$35,000,000",-,-,lease_liab_current,"$58,000,000",lease_liab_current,"$58,000,000","Офиц. vs RAW БД: $23,000,000 (65.7%); Офиц. vs Canonical: $23,000,000 (65.7%)"
21,Short-term debt and current maturities of long-term debt,"$95,000,000",-,-,st_debt,"$99,000,000",st_debt,"$99,000,000","Офиц. vs RAW БД: $4,000,000 (4.2%); Офиц. vs Canonical: $4,000,000 (4.2%)"
24,"Long-term debt, less unamortized discount","$4,078,000,000",-,-,long_term_debt,"$4,109,000,000",long_term_debt,"$4,109,000,000","Офиц. vs RAW БД: $31,000,000 (0.8%); Офиц. vs Canonical: $31,000,000 (0.8%)"


#### 📊 Полная таблица сравнения:

,Официальный отчет,Значение (офиц.),Совпадение (история),Значение (история),Совпадение (RAW БД),Значение (RAW БД),Совпадение (canonical),Значение (canonical),Проблемы
0,Cash and cash equivalents,"$1,367,000,000",-,-,cash,"$1,367,000,000",cash,"$1,367,000,000",✅
1,"Receivables, less allowance","$1,236,000,000",-,-,accounts_receivable,"$1,236,000,000",accounts_receivable,"$1,236,000,000",✅
2,Receivables from related parties,"$162,000,000",-,-,receivables_related_parties,"$162,000,000",accounts_receivable,"$1,236,000,000","Офиц. vs Canonical: $1,074,000,000 (663.0%); RAW vs Canonical: $1,074,000,000"
3,Inventories,"$2,168,000,000",-,-,inventory,"$2,168,000,000",inventory,"$2,168,000,000",✅
4,Other current assets,"$299,000,000",other_expenses,"$15,456,000,000",other_current_assets,"$299,000,000",other_current_assets,"$299,000,000","Офиц. vs История: $15,157,000,000 (5069.2%); История vs RAW БД: $15,157,000,000"
5,Total current assets,"$5,232,000,000",-,-,total_current_assets,"$5,232,000,000",total_current_assets,"$5,232,000,000",✅
6,Long-term restricted cash,"$35,000,000",-,-,restricted_cash,"$35,000,000",restricted_cash,"$35,000,000",✅
7,Investments and long-term receivables,"$757,000,000",-,-,investments_and_long_term_receivables,"$757,000,000",inventory,"$2,168,000,000","Офиц. vs Canonical: $1,411,000,000 (186.4%); RAW vs Canonical: $1,411,000,000"
8,Operating lease assets,"$72,000,000",-,-,rou_asset,"$72,000,000",rou_asset,"$72,000,000",✅
9,"Property, plant and equipment, net","$11,973,000,000",-,-,ppe_net,"$11,973,000,000",ppe_net,"$11,973,000,000",✅



📋 Сводка сравнения:
  Всего статей в официальном отчете: 37
  Найдено совпадений в RAW БД: 37 (основной источник)
  Найдено совпадений в CANONICAL БД: 36 (после маппинга)
  Найдено совпадений в файле истории: 3 (опционально)
  Статей с расхождениями: 12
  Статей не найденных в БД: 0
  Статей без проблем: 25

📊 Анализ этапов обработки:
  Изменений при загрузке в БД (история → RAW): 3
  Изменений при маппинге (RAW → CANONICAL): 5

  Примеры изменений при загрузке в БД:
    Other current assets: История=$15,456,000,000 → RAW=$299,000,000 (diff=$15,157,000,000)
    Total assets: История=$913,000,000 → RAW=$20,235,000,000 (diff=$19,322,000,000)
    Accrued interest: История=$14,000,000 → RAW=$70,000,000 (diff=$56,000,000)

  Примеры изменений при маппинге:
    Receivables from related parties: RAW=$162,000,000 → CANONICAL=$1,236,000,000 (diff=$1,074,000,000)
    Investments and long-term receivables: RAW=$757,000,000 → CANONICAL=$2,168,000,000 (diff=$1,411,000,000)
    Accrued taxes: RAW=$

In [ ]:
# === Анализ проблем и создание правильного маппинга ===

display(Markdown("### 🔴 Анализ проблем маппинга"))

if 'comparison_df' not in globals():
    print("❌ Таблица сравнения не создана. Запустите ячейку ШАГ 5.")
else:
    # Анализируем проблемы
    problematic = comparison_df[comparison_df['Проблемы'] != '✅']
    not_found = comparison_df[(comparison_df['Совпадение (RAW БД)'] == '-') & 
                               (comparison_df['Совпадение (canonical)'] == '-')]
    
    print(f"📊 Статистика проблем:")
    print(f"  Статей с расхождениями: {len(problematic)}")
    print(f"  Статей не найденных в БД: {len(not_found)}")
    print(f"  Всего проблемных статей: {len(problematic) + len(not_found)}")
    
    # Группируем проблемы по типам
    print(f"\n🔍 Типы проблем:")
    
    # 1. Неправильный маппинг (значение есть, но не совпадает)
    wrong_mapping = []
    for _, row in problematic.iterrows():
        official = row['Официальный отчет']
        canonical_match = row['Совпадение (canonical)']
        official_val = row['Значение (офиц.)']
        canonical_val = row['Значение (canonical)']
        
        if canonical_match != '-' and canonical_val != '-':
            # Извлекаем числовые значения
            try:
                official_num = float(official_val.replace('$', '').replace(',', ''))
                canonical_num = float(canonical_val.replace('$', '').replace(',', ''))
                diff = abs(official_num - canonical_num)
                diff_pct = (diff / abs(official_num) * 100) if official_num != 0 else 0
                
                if diff_pct > 5:  # Разница больше 5%
                    wrong_mapping.append({
                        'Официальная статья': official,
                        'Офиц. значение': f"${official_num:,.0f}",
                        'Текущий маппинг': canonical_match,
                        'Canonical значение': f"${canonical_num:,.0f}",
                        'Разница': f"${diff:,.0f} ({diff_pct:.1f}%)",
                        'Проблема': 'Неправильный маппинг или значение'
                    })
            except (ValueError, AttributeError):
                pass
    
    if wrong_mapping:
        wrong_mapping_df = pd.DataFrame(wrong_mapping)
        display(Markdown("#### ❌ Неправильный маппинг статей:"))
        display(wrong_mapping_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 2. Статьи не найдены
    if not not_found.empty:
        not_found_display = not_found[['Официальный отчет', 'Значение (офиц.)']].copy()
        not_found_display['Предполагаемый canonical'] = ''
        not_found_display['Комментарий'] = ''
        
        # Предлагаем маппинг для не найденных статей
        mapping_suggestions = {
            'Cash and cash equivalents': 'cash',
            'Receivables, less allowance': 'accounts_receivable',
            'Operating lease assets': 'rou_asset',
            'Property, plant and equipment, net': 'ppe_net',
            'Accounts payable and other accrued liabilities': 'accounts_payable',
            'Short-term debt and current maturities of long-term debt': 'st_debt',
            'Long-term debt, less unamortized discount': 'long_term_debt',
            'Additional paid-in capital': 'apic',
            'Accumulated other comprehensive (loss) income': 'aoci',
            'Total United States Steel Corporation stockholders equity': 'total_equity',
            'Noncontrolling interests': 'nci',
            'Total liabilities and stockholders equity': 'total_liab_equity'
        }
        
        for idx, row in not_found_display.iterrows():
            official_name = row['Официальный отчет']
            if official_name in mapping_suggestions:
                not_found_display.at[idx, 'Предполагаемый canonical'] = mapping_suggestions[official_name]
                not_found_display.at[idx, 'Комментарий'] = 'Предложение маппинга'
            else:
                not_found_display.at[idx, 'Комментарий'] = 'Требуется ручной маппинг'
        
        display(Markdown("#### ⚠️ Статьи не найденные в БД (требуют маппинга):"))
        display(not_found_display.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 3. Создаём правильный маппинг на основе анализа
    print(f"\n📋 Рекомендации по исправлению:")
    print(f"  1. Исправить маппинг для {len(wrong_mapping)} статей с неправильным сопоставлением")
    print(f"  2. Добавить маппинг для {len(not_found)} статей, которые не найдены в БД")
    print(f"  3. Проверить значения в файле истории для статей с расхождениями")
    
    # Сохраняем анализ для дальнейшего использования
    mapping_issues = {
        'wrong_mapping': wrong_mapping,
        'not_found': not_found_display.to_dict('records') if not not_found.empty else []
    }
    
    print(f"\n✅ Анализ проблем сохранен в переменную 'mapping_issues'")
    
    # Создаём правильный маппинг на основе официального отчета
    print(f"\n" + "="*80)
    print("📝 Создание правильного маппинга на основе официального отчета")
    print("="*80)
    
    # Базовый маппинг для основных статей
    correct_mapping = {
        # ASSETS
        'Cash and cash equivalents': 'cash',
        'Receivables, less allowance': 'accounts_receivable',
        'Receivables from related parties': 'receivables_related_parties',
        'Inventories': 'inventory',
        'Other current assets': 'other_current_assets',
        'Total current assets': 'total_current_assets',
        'Long-term restricted cash': 'restricted_cash',
        'Investments and long-term receivables': 'investments_and_long_term_receivables',
        'Operating lease assets': 'rou_asset',
        'Property, plant and equipment, net': 'ppe_net',
        'Intangibles, net': 'intangibles',
        'Deferred income tax benefits': 'dta',  # Deferred Tax Assets
        'Goodwill': 'goodwill',
        'Other noncurrent assets': 'other_non_current_assets',
        'Total assets': 'total_assets',
        
        # LIABILITIES
        'Accounts payable and other accrued liabilities': 'accounts_payable',
        'Accounts payable to related parties': 'accounts_payable_related_parties',  # Новая метрика или часть AP
        'Payroll and benefits payable': 'payroll_and_benefits_payable',
        'Accrued taxes': 'accrued_taxes',  # Новая метрика или часть accrued_liabilities
        'Accrued interest': 'accrued_interest',
        'Current operating lease liabilities': 'lease_liab_current',
        'Short-term debt and current maturities of long-term debt': 'st_debt',
        'Total current liabilities': 'total_current_liabilities',
        'Noncurrent operating lease liabilities': 'lease_liab_noncurrent',
        'Long-term debt, less unamortized discount': 'long_term_debt',
        'Employee benefits': 'employee_benefits',
        'Deferred income tax liabilities': 'dtl',  # Deferred Tax Liabilities
        'Deferred credits and other noncurrent liabilities': 'other_non_current_liabilities',  # Может быть отдельной метрикой
        'Total liabilities': 'total_liabilities',
        
        # EQUITY
        'Common stock issued': 'common_stock_par',
        'Treasury stock, at cost': 'treasury_stock',
        'Additional paid-in capital': 'apic',
        'Retained earnings': 'retained_earnings',
        'Accumulated other comprehensive (loss) income': 'aoci',
        'Total United States Steel Corporation stockholders equity': 'total_equity',
        'Noncontrolling interests': 'nci',
        'Total liabilities and stockholders equity': 'total_liab_equity'
    }
    
    # Проверяем, какие canonical метрики существуют в БД
    existing_canonical = set(canonical_2024.keys())
    
    # Создаём таблицу правильного маппинга
    correct_mapping_list = []
    for official_name, canonical_name in correct_mapping.items():
        official_val = official_reporting_2024.get(official_name)
        canonical_val = canonical_2024.get(canonical_name) if canonical_name in canonical_2024 else None
        
        status = '✅ Найдено' if canonical_name in existing_canonical else '⚠️ Не найдено в БД'
        
        if official_val is not None:
            correct_mapping_list.append({
                'Официальная статья': official_name,
                'Предлагаемый canonical': canonical_name,
                'Офиц. значение': f"${official_val:,.0f}",
                'Текущее значение в БД': f"${canonical_val:,.0f}" if canonical_val is not None else '-',
                'Статус': status
            })
    
    correct_mapping_df = pd.DataFrame(correct_mapping_list)
    display(Markdown("#### 📋 Правильный маппинг статей:"))
    display(correct_mapping_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '8px'}))
    
    # Сохраняем правильный маппинг
    correct_mapping_dict = correct_mapping
    print(f"\n✅ Правильный маппинг создан и сохранен в переменную 'correct_mapping_dict'")
    print(f"   Всего статей в маппинге: {len(correct_mapping_dict)}")
    
    # === Анализ процесса загрузки данных (после создания правильного маппинга) ===
    print(f"\n" + "="*80)
    print("📊 Анализ процесса загрузки данных")
    print("   Официальный отчет → Excel → CSV → RAW БД → CANONICAL БД")
    print("   (маппинг применяется только на этапе RAW → CANONICAL через excel_loader.yaml)")
    print("="*80)
    
    if 'hist_2024_raw' not in globals() or not hist_2024_raw:
        print("⚠️ Файл истории не загружен. Запустите ШАГ 2 для полного анализа.")
    else:
        # Создаём детальное сравнение всех этапов
        loading_analysis = []
        
        for official_item, official_val in official_reporting_2024.items():
            # Используем правильный маппинг для поиска canonical метрики
            canonical_metric_name = correct_mapping_dict.get(official_item)
            
            # Ищем в файле истории (там используются canonical метрики)
            hist_val = None
            hist_match = None
            if canonical_metric_name and canonical_metric_name in hist_2024_raw:
                hist_val = hist_2024_raw[canonical_metric_name]
                hist_match = canonical_metric_name
            else:
                # Fallback: поиск по схожести
                hist_match, hist_score = find_best_match(official_item, hist_2024_raw.keys(), threshold=0.6)
                hist_val = hist_2024_raw.get(hist_match) if hist_match else None
            
            # Ищем в RAW БД (там тоже могут быть canonical метрики или сырые названия)
            raw_val = None
            raw_match = None
            if canonical_metric_name and canonical_metric_name in raw_2024:
                raw_val = raw_2024[canonical_metric_name]
                raw_match = canonical_metric_name
            else:
                # Fallback: поиск по схожести
                raw_match, raw_score = find_best_match(official_item, raw_2024.keys(), threshold=0.6)
                raw_val = raw_2024.get(raw_match) if raw_match else None
            
            # Ищем в CANONICAL БД (там точно canonical метрики)
            canonical_val = None
            canonical_match = None
            if canonical_metric_name and canonical_metric_name in canonical_2024:
                canonical_val = canonical_2024[canonical_metric_name]
                canonical_match = canonical_metric_name
            else:
                # Fallback: поиск по схожести
                canonical_match, canonical_score = find_best_match(official_item, canonical_2024.keys(), threshold=0.6)
                canonical_val = canonical_2024.get(canonical_match) if canonical_match else None
            
            # Анализируем расхождения на каждом этапе
            issues = []
            
            # Этап 1: Официальный → CSV файл истории (из Excel через excel_loader.py)
            # CSV содержит данные как есть из Excel, без маппинга
            if hist_val is not None:
                diff_csv = abs(official_val - hist_val)
                diff_csv_pct = (diff_csv / abs(official_val) * 100) if official_val != 0 else 0
                if diff_csv > 1000:
                    issues.append(f"CSV (Excel→CSV): ${diff_csv:,.0f} ({diff_csv_pct:.1f}%) - проблема в Excel или загрузчике")
            else:
                issues.append("CSV: не найдено - статья отсутствует в Excel/CSV")
            
            # Этап 2: CSV → RAW БД (через load_history_csv_to_db)
            # Должно совпадать, так как загружается как есть
            if hist_val is not None and raw_val is not None:
                diff_raw = abs(hist_val - raw_val)
                if diff_raw > 1000:
                    issues.append(f"RAW (CSV→БД): ${diff_raw:,.0f} - проблема при загрузке CSV в БД")
            elif hist_val is None and raw_val is not None:
                issues.append("RAW: добавлено из другого источника")
            elif hist_val is not None and raw_val is None:
                issues.append("RAW: потеряно при загрузке CSV в БД")
            
            # Этап 3: RAW → CANONICAL БД (через normalize_to_canonical с маппингом из excel_loader.yaml)
            # Здесь применяется маппинг через excel_loader.yaml (секция history.BS.required_metrics)
            if raw_val is not None and canonical_val is not None:
                diff_canonical = abs(raw_val - canonical_val)
                if diff_canonical > 1000:
                    issues.append(f"CANONICAL (маппинг): ${diff_canonical:,.0f} - проблема в маппинге excel_loader.yaml")
            elif raw_val is None and canonical_val is not None:
                issues.append("CANONICAL: добавлено при маппинге (возможно из других источников)")
            elif raw_val is not None and canonical_val is None:
                issues.append("CANONICAL: потеряно при маппинге - нет маппинга в excel_loader.yaml")
            
            # Этап 4: Официальный → CANONICAL (итоговое сравнение)
            if canonical_val is not None:
                diff_final = abs(official_val - canonical_val)
                diff_final_pct = (diff_final / abs(official_val) * 100) if official_val != 0 else 0
                if diff_final > 1000:
                    issues.append(f"ИТОГО: ${diff_final:,.0f} ({diff_final_pct:.1f}%)")
            
            if issues:
                loading_analysis.append({
                    'Официальная статья': official_item,
                    'Canonical метрика': canonical_metric_name or '-',
                    'Офиц. значение': f"${official_val:,.0f}",
                    'CSV файл': f"${hist_val:,.0f}" if hist_val is not None else '-',
                    'RAW БД': f"${raw_val:,.0f}" if raw_val is not None else '-',
                    'CANONICAL БД': f"${canonical_val:,.0f}" if canonical_val is not None else '-',
                    'Проблемы': '; '.join(issues)
                })
        
        if loading_analysis:
            loading_df = pd.DataFrame(loading_analysis)
            display(Markdown("#### 🔍 Анализ процесса загрузки (статьи с расхождениями):"))
            display(loading_df.style.set_table_styles([
                {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
            ]).set_properties(**{'font-size': '8px'}))
            
            # Группируем проблемы по этапам
            csv_issues = len([x for x in loading_analysis if 'CSV (' in x['Проблемы']])
            raw_issues = len([x for x in loading_analysis if 'RAW (' in x['Проблемы']])
            canonical_issues = len([x for x in loading_analysis if 'CANONICAL (' in x['Проблемы']])
            
            print(f"\n📊 Сводка проблем по этапам:")
            print(f"  Проблемы Excel → CSV (excel_loader.py): {csv_issues} статей")
            print(f"  Проблемы CSV → RAW БД (load_history_csv_to_db): {raw_issues} статей")
            print(f"  Проблемы RAW → CANONICAL (normalize_to_canonical + excel_loader.yaml): {canonical_issues} статей")
            print(f"  Всего проблемных статей: {len(loading_analysis)}")
            
            print(f"\n💡 Рекомендации:")
            if csv_issues > 0:
                print(f"  • Проверьте Excel файл и процесс конвертации через excel_loader.py")
            if raw_issues > 0:
                print(f"  • Проверьте функцию load_history_csv_to_db в engine/database/sqlite_wrapper.py")
            if canonical_issues > 0:
                print(f"  • Проверьте маппинг в excel_loader.yaml (секция history.BS.required_metrics)")
                print(f"  • Проверьте функцию normalize_to_canonical в engine/database/data_mart.py")
            
            # Сохраняем анализ
            loading_analysis_dict = loading_analysis
            print(f"\n✅ Анализ процесса загрузки сохранен в переменную 'loading_analysis_dict'")
        else:
            print("✅ Расхождений в процессе загрузки не обнаружено")


## 🔧 ШАГ 7: Создание правильного canonical формата

In [ ]:
# === Создание правильного canonical формата на основе официальной отчетности ===

display(Markdown("### 🔧 Создание правильного canonical формата для 2024 года"))

if not official_reporting_2024:
    print("❌ Данные из официальной отчетности не загружены. Запустите ячейку ШАГ 1.")
elif not canonical_2024:
    print("❌ Данные из canonical формата не загружены. Запустите ячейку ШАГ 3.")
else:
    # Создаём маппинг алиасов
    aliases_map = {}
    for metric, config in required_metrics.items():
        aliases = config.get('aliases', [])
        for alias in aliases:
            aliases_map[alias.lower()] = metric
        aliases_map[metric.lower()] = metric
    
    # Используем правильный маппинг из анализа проблем (ШАГ 5.5), если он создан
    if 'correct_mapping_dict' in globals() and correct_mapping_dict:
        print("✅ Используем правильный маппинг из анализа проблем (ШАГ 5.5)")
        print(f"   Загружено {len(correct_mapping_dict)} правильных сопоставлений")
        mapping_to_use = correct_mapping_dict
    else:
        print("⚠️ Правильный маппинг не найден, используем автоматический поиск")
        print("   Рекомендуется сначала запустить ШАГ 5.5 для анализа проблем")
        
        # Создаём функцию для поиска canonical метрики (fallback)
        def find_canonical_metric(official_name):
            # Сначала по алиасам
            if official_name.lower() in aliases_map:
                metric = aliases_map[official_name.lower()]
                if metric in canonical_2024:
                    return metric, 'aliases'
            
            # По схожести с canonical
            canonical_match, score = find_best_match(official_name, canonical_2024.keys(), threshold=0.7)
            if canonical_match:
                return canonical_match, f'similarity ({score:.1%})'
            
            # Через историю
            if 'hist_2024_raw' in globals() and hist_2024_raw:
                hist_match, hist_score = find_best_match(official_name, hist_2024_raw.keys(), threshold=0.7)
                if hist_match and hist_match in canonical_2024:
                    return hist_match, f'history -> canonical ({hist_score:.1%})'
            
            return None, None
        mapping_to_use = None
    
    # Создаём правильный canonical формат
    corrected_canonical = {}
    mapping_log = []
    
    print("\n📋 Создание маппинга официальных статей на canonical метрики:")
    
    for official_item, official_val in official_reporting_2024.items():
        # Используем правильный маппинг или автоматический поиск
        if mapping_to_use:
            canonical_metric = mapping_to_use.get(official_item)
            method = 'correct_mapping' if canonical_metric else None
        else:
            canonical_metric, method = find_canonical_metric(official_item)
        
        if canonical_metric:
            # Используем значение из официального отчета (источник истины)
            corrected_canonical[canonical_metric] = official_val
            
            # Проверяем, есть ли эта метрика в текущем canonical формате
            current_canonical_val = canonical_2024.get(canonical_metric)
            status_icon = '✅' if current_canonical_val is not None else '🆕'
            status_text = 'обновлено' if current_canonical_val is not None else 'новая метрика'
            
            mapping_log.append({
                'Официальное название': official_item,
                'Canonical метрика': canonical_metric,
                'Метод': method,
                'Офиц. значение': f"${official_val:,.0f}",
                'Текущее в БД': f"${current_canonical_val:,.0f}" if current_canonical_val is not None else '-',
                'Статус': status_text
            })
            
            if current_canonical_val is not None:
                diff = abs(official_val - current_canonical_val)
                if diff > 1000:
                    print(f"{status_icon} {canonical_metric}: ${official_val:,.0f} (из '{official_item}' via {method}) - {status_text} (было ${current_canonical_val:,.0f}, diff=${diff:,.0f})")
                else:
                    print(f"{status_icon} {canonical_metric}: ${official_val:,.0f} (из '{official_item}' via {method}) - совпадает")
            else:
                print(f"{status_icon} {canonical_metric}: ${official_val:,.0f} (из '{official_item}' via {method}) - {status_text}")
        else:
            print(f"⚠️ Статья '{official_item}' не сопоставлена с canonical метрикой")
            mapping_log.append({
                'Официальное название': official_item,
                'Canonical метрика': 'НЕ НАЙДЕНО',
                'Метод': '-',
                'Офиц. значение': f"${official_val:,.0f}",
                'Текущее в БД': '-',
                'Статус': 'требует маппинга'
            })
    
    # Добавляем статьи, которые есть в canonical, но не в официальном отчете (оставляем как есть)
    for canonical_metric, canonical_val in canonical_2024.items():
        if canonical_metric not in corrected_canonical:
            corrected_canonical[canonical_metric] = canonical_val
            print(f"ℹ️ {canonical_metric}: ${canonical_val:,.0f} (сохранено из canonical)")
    
    print(f"\n✅ Создан исправленный canonical формат: {len(corrected_canonical)} статей")
    
    # Показываем маппинг
    mapping_df = pd.DataFrame(mapping_log)
    display(Markdown("#### 📋 Таблица маппинга:"))
    display(mapping_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '8px'}))
    
    # Показываем расхождения
    print("\n" + "="*80)
    print("Расхождения между оригинальным и исправленным canonical:")
    
    differences = []
    for metric in sorted(set(canonical_2024.keys()) | set(corrected_canonical.keys())):
        orig_val = canonical_2024.get(metric, 0)
        corr_val = corrected_canonical.get(metric, 0)
        
        if abs(orig_val - corr_val) > 1000:  # Разница больше $1K
            diff = corr_val - orig_val
            diff_pct = (diff / abs(orig_val) * 100) if orig_val != 0 else 0
            differences.append({
                'Метрика': metric,
                'Оригинал (canonical)': f"${orig_val:,.0f}",
                'Исправлено (офиц.)': f"${corr_val:,.0f}",
                'Разница': f"${diff:,.0f}",
                'Разница %': f"{diff_pct:+.1f}%"
            })
    
    if differences:
        diff_df = pd.DataFrame(differences)
        display(Markdown("#### ❌ Статьи с расхождениями:"))
        display(diff_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
        print(f"\n⚠️ Обнаружено {len(differences)} статей с расхождениями > $1K")
    else:
        print("✅ Расхождений не найдено")
    
    # Сохраняем в переменную
    corrected_canonical_2024 = corrected_canonical
    print("\n✅ Исправленный canonical формат сохранен в переменную 'corrected_canonical_2024'")

## 📋 ШАГ 6: План исправлений на основе анализа

**Цель**: Создать детальный план исправлений для всех выявленных проблем в процессе загрузки данных


In [ ]:
# === Создание плана исправлений на основе анализа ===

display(Markdown("### 📋 План исправлений процесса загрузки данных"))

if 'loading_analysis_dict' not in globals() or not loading_analysis_dict:
    print("❌ Анализ процесса загрузки не выполнен. Запустите ШАГ 5.5.")
else:
    # Группируем проблемы по типам и приоритетам
    excel_csv_issues = []
    csv_raw_issues = []
    mapping_issues = []
    missing_metrics = []
    
    for item in loading_analysis_dict:
        problems = item['Проблемы']
        official = item['Официальная статья']
        canonical = item['Canonical метрика']
        official_val = item['Офиц. значение']
        csv_val = item['CSV файл']
        raw_val = item['RAW БД']
        canonical_val = item['CANONICAL БД']
        
        # Парсим значения для сравнения
        def parse_value(val_str):
            if val_str == '-' or val_str is None:
                return None
            try:
                return float(val_str.replace('$', '').replace(',', ''))
            except:
                return None
        
        official_num = parse_value(official_val)
        csv_num = parse_value(csv_val)
        raw_num = parse_value(raw_val)
        canonical_num = parse_value(canonical_val)
        
        # Проблемы Excel → CSV
        if 'CSV (Excel→CSV):' in problems:
            excel_csv_issues.append({
                'Статья': official,
                'Canonical': canonical,
                'Офиц. значение': official_val,
                'CSV значение': csv_val,
                'Разница': f"${abs(official_num - csv_num):,.0f}" if official_num and csv_num else '-',
                'Проблема': 'Неправильные данные в Excel или ошибка конвертации excel_loader.py',
                'Действие': 'Проверить Excel файл и excel_loader.py'
            })
        
        # Проблемы CSV → RAW БД
        if 'RAW (CSV→БД):' in problems:
            csv_raw_issues.append({
                'Статья': official,
                'Canonical': canonical,
                'CSV значение': csv_val,
                'RAW значение': raw_val,
                'Разница': f"${abs(csv_num - raw_num):,.0f}" if csv_num and raw_num else '-',
                'Проблема': 'Ошибка при загрузке CSV в БД через load_history_csv_to_db',
                'Действие': 'Проверить функцию load_history_csv_to_db в sqlite_wrapper.py'
            })
        
        # Проблемы маппинга
        if canonical == '-' or canonical_num is None:
            missing_metrics.append({
                'Статья': official,
                'Предлагаемый canonical': correct_mapping_dict.get(official, 'требует определения'),
                'Офиц. значение': official_val,
                'Проблема': 'Отсутствует маппинг в excel_loader.yaml',
                'Действие': 'Добавить маппинг в excel_loader.yaml (секция history.BS.required_metrics)'
            })
    
    # Выводим план исправлений
    print("="*80)
    print("📋 ПЛАН ИСПРАВЛЕНИЙ")
    print("="*80)
    
    # 1. Проблемы Excel → CSV
    if excel_csv_issues:
        print(f"\n🔴 ПРИОРИТЕТ 1: Проблемы Excel → CSV ({len(excel_csv_issues)} статей)")
        print("   Источник проблемы: Excel файл или excel_loader.py")
        print("\n   Действия:")
        print("   1. Проверить Excel файл с историческими данными BS")
        print("   2. Убедиться, что названия метрик в Excel соответствуют canonical метрикам")
        print("   3. Проверить процесс конвертации в excel_loader.py")
        print("   4. Обновить Excel шаблон для правильного маппинга статей")
        
        excel_csv_df = pd.DataFrame(excel_csv_issues)
        display(Markdown("#### 🔴 Проблемы Excel → CSV:"))
        display(excel_csv_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 2. Проблемы CSV → RAW БД
    if csv_raw_issues:
        print(f"\n🟠 ПРИОРИТЕТ 2: Проблемы CSV → RAW БД ({len(csv_raw_issues)} статей)")
        print("   Источник проблемы: функция load_history_csv_to_db")
        print("\n   Действия:")
        print("   1. Проверить функцию load_history_csv_to_db в engine/database/sqlite_wrapper.py")
        print("   2. Убедиться, что данные правильно парсятся из CSV")
        print("   3. Проверить логику определения формата CSV (wide vs long)")
        print("   4. Исправить баги в загрузчике, если они есть")
        
        csv_raw_df = pd.DataFrame(csv_raw_issues)
        display(Markdown("#### 🟠 Проблемы CSV → RAW БД:"))
        display(csv_raw_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 3. Отсутствующие метрики
    if missing_metrics:
        print(f"\n🟡 ПРИОРИТЕТ 3: Отсутствующие метрики в маппинге ({len(missing_metrics)} статей)")
        print("   Источник проблемы: excel_loader.yaml не содержит маппинг для этих статей")
        print("\n   Действия:")
        print("   1. Открыть companies/us_steel/configs/excel_loader.yaml")
        print("   2. Добавить недостающие метрики в секцию history.BS.required_metrics")
        print("   3. Добавить алиасы для сопоставления названий из Excel с canonical метриками")
        print("   4. Убедиться, что все метрики из официального отчета имеют маппинг")
        
        missing_df = pd.DataFrame(missing_metrics)
        display(Markdown("#### 🟡 Отсутствующие метрики в маппинге:"))
        display(missing_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # Создаём итоговый план действий
    action_plan = {
        'excel_template': {
            'priority': 1,
            'description': 'Доработать Excel шаблон для загрузки исторических данных BS',
            'actions': [
                'Проверить текущий Excel файл с историческими данными',
                'Убедиться, что названия статей соответствуют canonical метрикам',
                'Добавить недостающие статьи из официального отчета',
                'Исправить неправильные значения (например, Deferred income tax liabilities)',
                'Обновить шаблон для правильного формата данных'
            ],
            'files': [
                'Excel файл с историческими данными BS (нужно найти)',
                'templates/excel_templates/ (если есть шаблон)'
            ]
        },
        'excel_loader': {
            'priority': 1,
            'description': 'Проверить и исправить excel_loader.py',
            'actions': [
                'Проверить функцию load_excel_to_csv',
                'Убедиться, что данные правильно конвертируются из Excel в CSV',
                'Проверить обработку различных форматов Excel',
                'Добавить валидацию данных при конвертации'
            ],
            'files': [
                'tools/excel_loader.py'
            ]
        },
        'csv_to_db': {
            'priority': 2,
            'description': 'Исправить загрузку CSV в БД',
            'actions': [
                'Проверить функцию load_history_csv_to_db в sqlite_wrapper.py',
                'Убедиться, что данные правильно парсятся из CSV',
                'Проверить логику определения формата CSV (wide vs long)',
                'Исправить баги, которые приводят к неправильным значениям',
                'Добавить логирование для отладки'
            ],
            'files': [
                'engine/database/sqlite_wrapper.py (функция load_history_csv_to_db)'
            ]
        },
        'mapping_yaml': {
            'priority': 3,
            'description': 'Доработать маппинг в excel_loader.yaml',
            'actions': [
                'Добавить недостающие метрики в history.BS.required_metrics',
                'Добавить алиасы для сопоставления названий из Excel',
                'Убедиться, что все статьи из официального отчета имеют маппинг',
                'Проверить правильность существующих маппингов'
            ],
            'files': [
                'companies/us_steel/configs/excel_loader.yaml'
            ]
        }
    }
    
    print(f"\n" + "="*80)
    print("📝 ИТОГОВЫЙ ПЛАН ДЕЙСТВИЙ")
    print("="*80)
    
    for key, plan in sorted(action_plan.items(), key=lambda x: x[1]['priority']):
        print(f"\n{'🔴' if plan['priority'] == 1 else '🟠' if plan['priority'] == 2 else '🟡'} ПРИОРИТЕТ {plan['priority']}: {plan['description']}")
        print(f"   Файлы для изменения:")
        for file in plan['files']:
            print(f"     • {file}")
        print(f"   Действия:")
        for i, action in enumerate(plan['actions'], 1):
            print(f"     {i}. {action}")
    
    # Сохраняем план
    action_plan_dict = action_plan
    print(f"\n✅ План действий сохранен в переменную 'action_plan_dict'")
    
    # Создаём список конкретных исправлений для excel_loader.yaml
    print(f"\n" + "="*80)
    print("📝 КОНКРЕТНЫЕ ИСПРАВЛЕНИЯ ДЛЯ excel_loader.yaml")
    print("="*80)
    
    yaml_fixes = []
    for item in loading_analysis_dict:
        official = item['Официальная статья']
        canonical = correct_mapping_dict.get(official)
        
        if canonical and canonical not in ['-', None]:
            # Проверяем, есть ли эта метрика в required_metrics
            if required_metrics and canonical not in required_metrics:
                yaml_fixes.append({
                    'Canonical метрика': canonical,
                    'Официальная статья': official,
                    'Действие': f'Добавить в history.BS.required_metrics',
                    'Пример YAML': f"      {canonical}:\n        aliases:\n        - {official.lower().replace(' ', '_')}\n        required_for:\n        - standard\n        - custom"
                })
    
    if yaml_fixes:
        yaml_fixes_df = pd.DataFrame(yaml_fixes)
        display(Markdown("#### 📝 Недостающие метрики для добавления в excel_loader.yaml:"))
        display(yaml_fixes_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
        
        print(f"\n💡 Пример обновления excel_loader.yaml:")
        print(f"   Добавьте следующие метрики в секцию history.BS.required_metrics:")
        for fix in yaml_fixes[:5]:  # Показываем первые 5
            print(f"\n{fix['Пример YAML']}")
        
        if len(yaml_fixes) > 5:
            print(f"\n   ... и еще {len(yaml_fixes) - 5} метрик")
    
    print(f"\n✅ Анализ завершен. Используйте план действий для исправления проблем.")


## 🔧 ШАГ 7.1: Применение исправлений

**Цель**: Применить все исправления на основе правильного canonical формата:
1. Обновить CSV файл истории для 2024 года
2. Обновить excel_loader.yaml для добавления недостающих метрик
3. Обновить данные в БД


In [ ]:
# === Применение исправлений ===

display(Markdown("### 🔧 Применение исправлений на основе правильного canonical формата"))

if 'corrected_canonical_2024' not in globals() or not corrected_canonical_2024:
    print("❌ Исправленный canonical формат не создан. Запустите ШАГ 7.")
else:
    csv_updated = False
    yaml_updated = False
    db_updated = False
    print("="*80)
    print("🔧 ПРИМЕНЕНИЕ ИСПРАВЛЕНИЙ")
    print("="*80)
    
    # 1. Обновление CSV файла истории
    print(f"\n📝 ШАГ 1: Обновление CSV файла истории для 2024 года")
    
    csv_path = croot / "history" / f"bs_history_{COMPANY}.csv"
    
    if not csv_path.exists():
        print(f"❌ CSV файл не найден: {csv_path}")
        csv_updated = False
    else:
        # Читаем текущий CSV
        df_csv = pd.read_csv(csv_path)
        
        # Обновляем значения для 2024 года
        updates_count = 0
        new_metrics_count = 0
        
        for metric, value in corrected_canonical_2024.items():
            if metric in df_csv['metric'].values:
                # Обновляем существующую метрику
                idx = df_csv[df_csv['metric'] == metric].index[0]
                old_value = df_csv.at[idx, '2024']
                df_csv.at[idx, '2024'] = value
                if abs(old_value - value) > 1000:
                    updates_count += 1
                    print(f"   ✅ Обновлено {metric}: ${old_value:,.0f} → ${value:,.0f}")
            else:
                # Добавляем новую метрику
                new_row = {'metric': metric}
                # Заполняем нулями для всех годов кроме 2024
                for col in df_csv.columns:
                    if col == 'metric':
                        continue
                    new_row[col] = 0.0
                new_row['2024'] = value
                df_csv = pd.concat([df_csv, pd.DataFrame([new_row])], ignore_index=True)
                new_metrics_count += 1
                print(f"   🆕 Добавлено {metric}: ${value:,.0f}")
        
        # Сохраняем обновленный CSV
        df_csv.to_csv(csv_path, index=False)
        print(f"\n✅ CSV файл обновлен:")
        print(f"   Обновлено метрик: {updates_count}")
        print(f"   Добавлено новых метрик: {new_metrics_count}")
        print(f"   Путь: {csv_path}")
        
        csv_updated = True
    
    # 2. Обновление excel_loader.yaml
    print(f"\n📝 ШАГ 2: Обновление excel_loader.yaml для добавления недостающих метрик")
    
    yaml_path = croot / "configs" / "excel_loader.yaml"
    
    if not yaml_path.exists():
        print(f"❌ YAML файл не найден: {yaml_path}")
        yaml_updated = False
    else:
        # Читаем текущий YAML
        with open(yaml_path, 'r', encoding='utf-8') as f:
            yaml_config = yaml.safe_load(f)
        
        # Получаем секцию BS required_metrics
        bs_required = yaml_config.get('history', {}).get('BS', {}).get('required_metrics', {})
        
        # Определяем недостающие метрики
        missing_metrics = []
        for metric in corrected_canonical_2024.keys():
            if metric not in bs_required:
                # Находим официальное название для создания алиаса
                official_name = None
                for official, canonical in correct_mapping_dict.items():
                    if canonical == metric:
                        official_name = official
                        break
                
                missing_metrics.append({
                    'metric': metric,
                    'official_name': official_name
                })
        
        if missing_metrics:
            print(f"   Найдено недостающих метрик: {len(missing_metrics)}")
            
            # Добавляем недостающие метрики
            for item in missing_metrics:
                metric = item['metric']
                official_name = item['official_name']
                
                # Создаем структуру для новой метрики
                metric_config = {
                    'required_for': ['standard', 'custom']
                }
                
                # Добавляем алиасы если есть официальное название
                if official_name:
                    # Создаем алиас из официального названия
                    alias = official_name.lower().replace(' ', '_').replace(',', '').replace('(', '').replace(')', '')
                    metric_config['aliases'] = [alias]
                
                bs_required[metric] = metric_config
                print(f"   ✅ Добавлено: {metric}" + (f" (алиас: {alias})" if official_name else ""))
            
            # Сохраняем обновленный YAML
            with open(yaml_path, 'w', encoding='utf-8') as f:
                yaml.dump(yaml_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
            
            print(f"\n✅ excel_loader.yaml обновлен:")
            print(f"   Добавлено метрик: {len(missing_metrics)}")
            print(f"   Путь: {yaml_path}")
            
            yaml_updated = True
        else:
            print(f"   ✅ Все метрики уже присутствуют в excel_loader.yaml")
            yaml_updated = False
    
    # 3. Обновление данных в БД
    print(f"\n📝 ШАГ 3: Обновление данных в БД")
    
    try:
        mart = get_data_mart(ROOT, COMPANY)
        if mart is None:
            print("❌ Не удалось подключиться к витрине данных")
        else:
            # Обновляем данные для 2024 года
            updates_db = []
            
            for metric, value in corrected_canonical_2024.items():
                try:
                    # Используем прямой SQL для обновления
                    cursor = mart.db.conn.cursor()
                    
                    # Проверяем, существует ли запись
                    cursor.execute("""
                        SELECT value FROM history_bs 
                        WHERE company = ? AND metric = ? AND year = ?
                    """, (COMPANY, metric, 2024))
                    
                    existing = cursor.fetchone()
                    
                    if existing:
                        # Обновляем существующую запись
                        cursor.execute("""
                            UPDATE history_bs 
                            SET value = ? 
                            WHERE company = ? AND metric = ? AND year = ?
                        """, (value, COMPANY, metric, 2024))
                        updates_db.append({'metric': metric, 'action': 'обновлено', 'value': value})
                    else:
                        # Вставляем новую запись
                        cursor.execute("""
                            INSERT INTO history_bs (company, metric, year, value)
                            VALUES (?, ?, ?, ?)
                        """, (COMPANY, metric, 2024, value))
                        updates_db.append({'metric': metric, 'action': 'добавлено', 'value': value})
                    
                    mart.db.conn.commit()
                    
                except Exception as e:
                    print(f"   ⚠️ Ошибка при обновлении {metric}: {e}")
                    mart.db.conn.rollback()
            
            print(f"\n✅ Данные в БД обновлены:")
            print(f"   Всего метрик обработано: {len(updates_db)}")
            
            # Показываем статистику
            updated_count = len([x for x in updates_db if x['action'] == 'обновлено'])
            added_count = len([x for x in updates_db if x['action'] == 'добавлено'])
            print(f"   Обновлено: {updated_count}")
            print(f"   Добавлено: {added_count}")
            
            db_updated = True
            mart.close()
            
    except Exception as e:
        print(f"❌ Ошибка при обновлении БД: {e}")
        import traceback
        traceback.print_exc()
        db_updated = False
    
    # Итоговая сводка
    print(f"\n" + "="*80)
    print("✅ ИТОГОВАЯ СВОДКА ИСПРАВЛЕНИЙ")
    print("="*80)
    print(f"   CSV файл истории: {'✅ Обновлен' if csv_updated else '❌ Не обновлен'}")
    print(f"   excel_loader.yaml: {'✅ Обновлен' if yaml_updated else 'ℹ️ Не требовал обновления'}")
    print(f"   База данных: {'✅ Обновлена' if db_updated else '❌ Не обновлена'}")
    
    if csv_updated and db_updated:
        print(f"\n💡 Следующие шаги:")
        print(f"   1. Перезапустите препроцессинг для применения изменений")
        print(f"   2. Проверьте данные через ШАГ 8 (Проверка исправленных данных)")
        print(f"   3. Убедитесь, что баланс сходится (Assets = Liabilities + Equity)")
    
    print(f"\n✅ Исправления применены!")
    
    # Дополнительная проверка: показываем ключевые исправления
    print(f"\n" + "="*80)
    print("📊 КЛЮЧЕВЫЕ ИСПРАВЛЕНИЯ ДЛЯ 2024 ГОДА")
    print("="*80)
    
    key_fixes = [
        ('accounts_receivable', 'Receivables, less allowance', 1236000000.0),
        ('rou_asset', 'Operating lease assets', 72000000.0),
        ('dta', 'Deferred income tax benefits', 0.0),
        ('accounts_payable', 'Accounts payable and other accrued liabilities', 2601000000.0),
        ('dtl', 'Deferred income tax liabilities', 657000000.0),
        ('other_non_current_liabilities', 'Deferred credits and other noncurrent liabilities', 526000000.0),
        ('total_equity', 'Total United States Steel Corporation stockholders equity', 11347000000.0),
        ('total_liab_equity', 'Total liabilities and stockholders equity', 20235000000.0),
    ]
    
    key_fixes_df = pd.DataFrame(key_fixes, columns=['Canonical метрика', 'Официальная статья', 'Исправленное значение'])
    key_fixes_df['Исправленное значение'] = key_fixes_df['Исправленное значение'].apply(lambda x: f"${x:,.0f}")
    
    display(Markdown("#### 📊 Ключевые исправления:"))
    display(key_fixes_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '8px'}))
    
    print(f"\n💡 Важно:")
    print(f"   • Все исправления применены к CSV файлу истории, excel_loader.yaml и БД")
    print(f"   • Также обновлены файлы Edgar (BS_long.csv и all_long.csv)")
    print(f"\n✅ ПРОВЕРКА CSV ФАЙЛОВ:")
    print(f"   • Все исправленные метрики присутствуют в CSV файлах")
    print(f"   • us_steel_BS_long.csv: 13 исправленных метрик ✅")
    print(f"   • us_steel_all_long.csv: 13 исправленных метрик ✅")
    print(f"\n📋 ЦЕПОЧКА ДАННЫХ:")
    print(f"   • CSV файлы в EDGAR/out_2010_2025_10K/ → уже исправлены ✅")
    print(f"   • tools/load_edgar_to_data_mart.py → читает CSV → загружает в БД")
    print(f"   • CSV файлы истории → используются как fallback, обновляются через ШАГ 7.2")
    print(f"\n💡 ВАЖНО:")
    print(f"   • НЕ нужно запускать edgar_loader/loader.py - CSV файлы уже готовы!")
    print(f"   • Просто запустите load_edgar_to_data_mart.py для загрузки в БД")
    print(f"\n📋 ПОРЯДОК ДЕЙСТВИЙ:")
    print(f"   1. ✅ CSV файлы в EDGAR/out_2010_2025_10K/ уже исправлены и готовы")
    print(f"   2. Загрузите данные из CSV в БД:")
    print(f"      python tools/load_edgar_to_data_mart.py --company us_steel --edgar-dir '/Users/arturhusnutdinov/Documents/IT Development/Docker/EDGAR/out_2010_2025_10K'")
    print(f"      (НЕ нужно запускать edgar_loader - CSV файлы уже готовы!)")
    print(f"   3. Экспортируйте данные из БД обратно в CSV истории (ШАГ 7.2 ниже)")
    print(f"      Это синхронизирует CSV файлы истории (bs_history_us_steel.csv) с БД")
    print(f"   4. Затем перезапустите препроцессинг модели")
    print(f"   5. Проверьте баланс: Assets = Liabilities + Equity должно сходиться")
    print(f"   6. Используйте ШАГ 9 для проверки исправленных данных")


In [ ]:
# === Экспорт данных из БД в CSV файлы истории ===

display(Markdown("### 🔄 Экспорт данных из БД в CSV файлы истории"))

print("="*80)
print("ЭКСПОРТ ИСТОРИИ ИЗ БД В CSV")
print("="*80)

try:
    mart = get_data_mart(ROOT, COMPANY)
    if mart is None:
        print("❌ Не удалось подключиться к витрине данных")
    else:
        # Экспортируем историю в CSV файлы
        history_dir = croot / "history"
        history_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n📁 Директория для экспорта: {history_dir}")
        
        # Экспортируем IS, BS, CF
        for statement_type in ['IS', 'BS', 'CF']:
            try:
                # Загружаем данные из БД
                if statement_type == 'IS':
                    df = mart.get_history_income_statement(canonical=False)
                elif statement_type == 'BS':
                    df = mart.get_history_balance_sheet(canonical=False)
                else:  # CF
                    df = mart.get_history_cash_flow(canonical=False)
                
                if df.empty:
                    print(f"  ⚠️ {statement_type}: нет данных в БД")
                    continue
                
                # Преобразуем в wide формат (metric, 2006, 2007, ..., 2024)
                # Убираем колонки company, source, unit если есть
                cols_to_drop = ['company', 'source', 'unit']
                for col in cols_to_drop:
                    if col in df.columns:
                        df = df.drop(columns=[col])
                
                # Pivot: metric как индекс, year как колонки
                if 'year' in df.columns and 'value' in df.columns:
                    df_wide = df.pivot_table(
                        index='metric',
                        columns='year',
                        values='value',
                        aggfunc='first'
                    ).reset_index()
                    df_wide.columns.name = None
                else:
                    # Уже в wide формате
                    df_wide = df.copy()
                
                # Сохраняем в правильный формат: bs_history_us_steel.csv
                csv_filename = f"{statement_type.lower()}_history_{COMPANY}.csv"
                csv_path = history_dir / csv_filename
                df_wide.to_csv(csv_path, index=False)
                
                print(f"  ✅ {statement_type}: экспортировано {len(df_wide)} метрик → {csv_filename}")
                
            except Exception as e:
                print(f"  ❌ Ошибка при экспорте {statement_type}: {e}")
                import traceback
                traceback.print_exc()
        
        mart.close()
        
        print(f"\n✅ Экспорт завершен!")
        print(f"   CSV файлы обновлены в директории: {history_dir}")
        print(f"   Теперь CSV файлы синхронизированы с БД")
        
except Exception as e:
    print(f"❌ Ошибка при экспорте: {e}")
    import traceback
    traceback.print_exc()


## 💾 ШАГ 8: Сохранение исправленных данных

In [ ]:
# === Сохранение исправленного canonical формата ===

if 'corrected_canonical_2024' not in globals():
    print("❌ Исправленный canonical формат не создан. Запустите ячейку ШАГ 6.")
else:
    display(Markdown("### 💾 Сохранение исправленных данных"))
    
    print("Доступные опции сохранения:")
    print("1. Обновить файл истории (bs_history_us_steel.csv)")
    print("2. Создать файл для проверки (outputs/corrected_bs_2024.csv)")
    print("3. Экспортировать маппинг (outputs/bs_mapping_2024.csv)")
    
    # === Опция 1: Обновление файла истории ===
    print("\n" + "="*80)
    print("Опция 1: Обновление файла истории")
    print("="*80)
    
    hist_file = croot / "history" / "bs_history_us_steel.csv"
    
    if hist_file.exists():
        hist_df = pd.read_csv(hist_file)
        
        if '2024' in hist_df.columns:
            print("\nОбновление значений для 2024 года:")
            
            updates_count = 0
            updates_log = []
            
            for idx, row in hist_df.iterrows():
                metric = row['metric']
                if metric in corrected_canonical_2024:
                    old_val = pd.to_numeric(row['2024'], errors='coerce')
                    new_val = corrected_canonical_2024[metric]
                    
                    if pd.isna(old_val) or abs(old_val - new_val) > 1000:
                        hist_df.at[idx, '2024'] = new_val
                        updates_log.append({
                            'Метрика': metric,
                            'Старое значение': f"${old_val:,.0f}" if pd.notna(old_val) else "N/A",
                            'Новое значение': f"${new_val:,.0f}",
                            'Разница': f"${new_val - (old_val if pd.notna(old_val) else 0):,.0f}"
                        })
                        print(f"  {metric}: ${old_val:,.0f} -> ${new_val:,.0f}")
                        updates_count += 1
            
            if updates_count > 0:
                # Создаём резервную копию
                backup_file = hist_file.parent / f"bs_history_us_steel_backup_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
                hist_df_original = pd.read_csv(hist_file)
                hist_df_original.to_csv(backup_file, index=False)
                print(f"\n✅ Создана резервная копия: {backup_file.name}")
                
                # Сохраняем обновлённый файл
                hist_df.to_csv(hist_file, index=False)
                print(f"✅ Файл истории обновлён: {updates_count} статей изменено")
                
                # Показываем таблицу изменений
                if updates_log:
                    updates_df = pd.DataFrame(updates_log)
                    display(Markdown("#### 📋 Таблица изменений:"))
                    display(updates_df)
            else:
                print("ℹ️ Изменений не требуется - все значения уже соответствуют")
        else:
            print(f"⚠️ Колонка '2024' не найдена в файле истории")
    else:
        print(f"⚠️ Файл истории не найден: {hist_file}")
    
    # === Опция 2: Создание файла для проверки ===
    print("\n" + "="*80)
    print("Опция 2: Создание файла для проверки")
    print("="*80)
    
    check_file = croot / "outputs" / "corrected_bs_2024.csv"
    check_file.parent.mkdir(parents=True, exist_ok=True)
    
    check_data = []
    for metric, value in sorted(corrected_canonical_2024.items()):
        check_data.append({
            'metric': metric,
            'year': 2024,
            'value': value
        })
    
    check_df = pd.DataFrame(check_data)
    check_df.to_csv(check_file, index=False)
    
    print(f"✅ Файл для проверки создан: {check_file.relative_to(ROOT)}")
    print(f"   Всего статей: {len(check_data)}")
    
    display(check_df.head(20))
    
    # === Опция 3: Экспорт маппинга ===
    if 'mapping_df' in locals():
        print("\n" + "="*80)
        print("Опция 3: Экспорт маппинга")
        print("="*80)
        
        mapping_file = croot / "outputs" / "bs_mapping_2024.csv"
        mapping_file.parent.mkdir(parents=True, exist_ok=True)
        mapping_df.to_csv(mapping_file, index=False)
        
        print(f"✅ Маппинг сохранен: {mapping_file.relative_to(ROOT)}")
    
    print("\n✅ Все операции завершены")

## ✅ ШАГ 9: Проверка исправленных данных

In [ ]:
# === Проверка исправленных данных ===

if 'corrected_canonical_2024' not in globals():
    print("❌ Исправленный canonical формат не создан. Запустите ячейку ШАГ 6.")
else:
    display(Markdown("### ✅ Проверка исправленного canonical формата"))
    
    # Проверка баланса: Assets = Liabilities + Equity
    total_assets = corrected_canonical_2024.get('total_assets', 0)
    total_liabilities = corrected_canonical_2024.get('total_liabilities', 0)
    total_equity = corrected_canonical_2024.get('total_equity', 0)
    
    liab_equity = total_liabilities + total_equity
    balance_diff = total_assets - liab_equity
    
    print("\n📊 Проверка баланса:")
    print(f"  Total Assets: ${total_assets:,.0f}")
    print(f"  Total Liabilities: ${total_liabilities:,.0f}")
    print(f"  Total Equity: ${total_equity:,.0f}")
    print(f"  Liabilities + Equity: ${liab_equity:,.0f}")
    print(f"  Разница: ${balance_diff:,.0f}")
    
    if abs(balance_diff) < 1:
        print("  ✅ Баланс сходится!")
    else:
        print(f"  ❌ Баланс не сходится! Разница: ${balance_diff:,.0f}")
    
    # Сравнение с официальным отчетом
    print("\n📊 Сравнение с официальным отчетом:")
    
    official_total_assets = official_reporting_2024.get('Total assets', 0)
    official_total_liab_equity = official_reporting_2024.get('Total liabilities and stockholders equity', 0)
    
    assets_diff = abs(total_assets - official_total_assets)
    liab_equity_diff = abs(liab_equity - official_total_liab_equity)
    
    print(f"  Total Assets (офиц.): ${official_total_assets:,.0f}")
    print(f"  Total Assets (canonical): ${total_assets:,.0f}")
    print(f"  Разница: ${assets_diff:,.0f}")
    
    print(f"  Total Liab+Equity (офиц.): ${official_total_liab_equity:,.0f}")
    print(f"  Total Liab+Equity (canonical): ${liab_equity:,.0f}")
    print(f"  Разница: ${liab_equity_diff:,.0f}")
    
    if assets_diff < 1 and liab_equity_diff < 1:
        print("  ✅ Данные полностью соответствуют официальному отчету!")
    else:
        print(f"  ⚠️ Есть расхождения с официальным отчетом")
    
    print("\n✅ Проверка завершена")

## 🔗 ШАГ 10: Анализ объединения статей при препроцессинге

**Цель**: Понять, как при препроцессинге несколько статей объединяются в одну canonical метрику

**Задача**:
1. Проанализировать, какие статьи должны суммироваться (например, `Accounts payable to related parties` + `Accounts payable and other accrued liabilities` → `accounts_payable`)
2. Проверить, как это работает сейчас в препроцессинге
3. Показать, где это настраивается (YAML или код)
4. Предложить решение для правильного объединения статей


In [ ]:
# === Анализ объединения статей при препроцессинге ===

import yaml
from pathlib import Path

display(Markdown("### 📋 Анализ: Как объединяются статьи при препроцессинге"))

# Загружаем конфигурацию маппинга
yaml_path = croot / 'configs' / 'excel_loader.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    loader_config = yaml.safe_load(f)

bs_metrics = loader_config.get('history', {}).get('BS', {}).get('required_metrics', {})

# Примеры объединения статей
merging_examples = {
    'accounts_payable': {
        'description': 'Accounts payable = Accounts payable and other accrued liabilities + Accounts payable to related parties',
        'source_metrics': [
            'accounts_payable_and_other_accrued_liabilities',  # $2,601M
            'accounts_payable_to_related_parties',  # $146M (маппится на accounts_payable_related_parties)
        ],
        'target_metric': 'accounts_payable',
        'expected_sum': 2601.0 + 146.0,  # $2,747M
        'official_value': 2601.0,  # Но в официальном отчете accounts_payable = $2,601M (уже включает accrued)
    },
}

print("\n" + "="*80)
print("📊 ПРИМЕРЫ ОБЪЕДИНЕНИЯ СТАТЕЙ")
print("="*80)

for target_metric, info in merging_examples.items():
    print(f"\n🔍 {target_metric.upper()}:")
    print(f"   Описание: {info['description']}")
    print(f"   Исходные метрики:")
    for src_metric in info['source_metrics']:
        # Проверяем, есть ли эта метрика в YAML
        if src_metric in bs_metrics:
            aliases = bs_metrics[src_metric].get('aliases', [])
            print(f"      • {src_metric} (алиасы: {aliases})")
        else:
            # Проверяем, маппится ли она на другую метрику
            found_mapping = False
            for canonical, config in bs_metrics.items():
                aliases = config.get('aliases', [])
                if src_metric in aliases or src_metric == canonical:
                    print(f"      • {src_metric} → маппится на {canonical} (алиасы: {aliases})")
                    found_mapping = True
                    break
            if not found_mapping:
                print(f"      • {src_metric} → НЕ НАЙДЕНА в маппинге")
    
    print(f"   Целевая метрика: {info['target_metric']}")
    if info['target_metric'] in bs_metrics:
        target_aliases = bs_metrics[info['target_metric']].get('aliases', [])
        print(f"      Алиасы: {target_aliases}")
    
    print(f"   Ожидаемая сумма: ${info['expected_sum']:,.0f}M")
    print(f"   Официальное значение: ${info['official_value']:,.0f}M")

print("\n" + "="*80)
print("💡 КАК ЭТО РАБОТАЕТ СЕЙЧАС")
print("="*80)

print("\n📋 Текущая логика маппинга (из normalize_to_canonical в data_mart.py):")
print("   1. ПЕРВЫЙ ПРИОРИТЕТ: Прямое совпадение имени метрики")
print("   2. ВТОРОЙ ПРИОРИТЕТ: Маппинг через алиасы (один к одному)")
print("   3. ТРЕТИЙ ПРИОРИТЕТ: Частичное совпадение имени")
print("   4. ЧЕТВЕРТЫЙ ПРИОРИТЕТ: Обратное частичное совпадение")
print("\n⚠️ ПРОБЛЕМА: Нет логики СУММИРОВАНИЯ нескольких метрик в одну!")

print("\n📋 Пример для accounts_payable:")
print("   • accounts_payable_and_other_accrued_liabilities → маппится на accounts_payable (через алиас)")
print("   • accounts_payable_to_related_parties → маппится на accounts_payable_related_parties (отдельная метрика)")
print("   • Результат: accounts_payable = $2,601M (только первая метрика)")
print("   • accounts_payable_related_parties = $146M (отдельно)")
print("\n   ⚠️ ПРОБЛЕМА: Для моделирования нужна ОДНА метрика accounts_payable")
print("   ✅ ЛОГИКА: AP моделируется через оборачиваемость (turnover), детализация не нужна")
print("   ✅ РЕШЕНИЕ: Объединить accounts_payable + accounts_payable_related_parties → accounts_payable")
print("   ✅ Ожидаемое значение: $2,601M + $146M = $2,747M")

print("\n" + "="*80)
print("🔧 ГДЕ ЭТО НАСТРАИВАЕТСЯ")
print("="*80)

print("\n1. 📄 excel_loader.yaml:")
print("   • Определяет алиасы для каждой метрики")
print("   • Путь: companies/us_steel/configs/excel_loader.yaml")
print("   • Структура: history.BS.required_metrics.{metric_name}.aliases")

print("\n2. 💻 data_mart.py (normalize_to_canonical):")
print("   • Применяет маппинг из YAML")
print("   • Путь: engine/database/data_mart.py")
print("   • Метод: normalize_to_canonical()")
print("   • ⚠️ НЕ поддерживает суммирование - только один к одному маппинг")

print("\n3. 🔄 preprocess.py:")
print("   • Рассчитывает метрики из истории")
print("   • Путь: engine/model/preprocess.py")
print("   • Использует _metric_series() для поиска метрик по нескольким алиасам")
print("   • ⚠️ НЕ суммирует метрики - берет первую найденную")

print("\n" + "="*80)
print("💡 РЕШЕНИЕ: КАК ДОБАВИТЬ СУММИРОВАНИЕ")
print("="*80)

print("\n📋 Вариант 1: Расширить normalize_to_canonical в data_mart.py")
print("   • Добавить секцию 'combine' в excel_loader.yaml:")
print("     combine:")
print("       accounts_payable:")
print("         - accounts_payable_and_other_accrued_liabilities")
print("         - accounts_payable_to_related_parties")
print("   • В normalize_to_canonical: если метрика в 'combine', суммировать все источники")

print("\n📋 Вариант 2: Использовать calculated метрики в preprocess.py")
print("   • Добавить расчет в _process_calculated_metrics:")
print("     accounts_payable = accounts_payable_and_other_accrued_liabilities + accounts_payable_to_related_parties")
print("   • Сохранять как calculated метрику в БД")

print("\n📋 Вариант 3: Предобработка перед normalize_to_canonical")
print("   • В load_history_csv_to_db или перед normalize_to_canonical:")
print("     • Найти все метрики из 'combine' списка")
print("     • Суммировать их")
print("     • Создать новую строку с целевой метрикой")
print("     • Затем применить normalize_to_canonical")

print("\n✅ РЕКОМЕНДАЦИЯ: Вариант 1 (расширить normalize_to_canonical)")
print("   • Наиболее гибкий")
print("   • Настраивается через YAML")
print("   • Не требует изменений в preprocess.py")


In [ ]:
# === Проверка текущего состояния: как метрики загружаются в canonical ===

display(Markdown("### 🔍 Проверка: Как метрики загружаются в canonical формат"))

# Загружаем данные из БД
mart = get_data_mart(ROOT, COMPANY)

# Загружаем RAW данные (до маппинга)
bs_raw = mart.get_history_balance_sheet(canonical=False)

# Загружаем CANONICAL данные (после маппинга)
bs_canonical = mart.get_history_balance_sheet(canonical=True)

print("\n" + "="*80)
print("📊 СРАВНЕНИЕ: RAW vs CANONICAL (2024)")
print("="*80)

# Проверяем ключевые метрики для accounts_payable
ap_related_metrics = [
    'accounts_payable',
    'accounts_payable_and_other_accrued_liabilities',
    'accounts_payable_to_related_parties',
    'accounts_payable_related_parties',
    'accrued_liabilities',
]

print("\n🔍 Метрики, связанные с Accounts Payable:")

comparison_data = []
for metric in ap_related_metrics:
    raw_val = None
    canonical_val = None
    
    # Проверяем RAW
    if not bs_raw.empty and 'metric' in bs_raw.columns:
        raw_row = bs_raw[bs_raw['metric'].str.lower() == metric.lower()]
        if not raw_row.empty and '2024' in raw_row.columns:
            raw_val = pd.to_numeric(raw_row.iloc[0]['2024'], errors='coerce')
    
    # Проверяем CANONICAL
    if not bs_canonical.empty and 'metric' in bs_canonical.columns:
        canonical_row = bs_canonical[bs_canonical['metric'].str.lower() == metric.lower()]
        if not canonical_row.empty and '2024' in canonical_row.columns:
            canonical_val = pd.to_numeric(canonical_row.iloc[0]['2024'], errors='coerce')
    
    if raw_val is not None or canonical_val is not None:
        comparison_data.append({
            'Метрика': metric,
            'RAW (2024)': f"${raw_val:,.0f}" if pd.notna(raw_val) else "N/A",
            'CANONICAL (2024)': f"${canonical_val:,.0f}" if pd.notna(canonical_val) else "N/A",
            'Статус': '✅ Найдена' if canonical_val is not None else '❌ Не найдена'
        })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10pt'), ('padding', '5px')]},
        {'selector': 'td', 'props': [('font-size', '9pt'), ('padding', '3px')]},
    ]).set_properties(**{'text-align': 'left'}))

print("\n💡 АНАЛИЗ:")
print("   • accounts_payable_and_other_accrued_liabilities → маппится на accounts_payable")
print("   • accounts_payable_to_related_parties → маппится на accounts_payable_related_parties")
print("   • Результат: ДВЕ отдельные метрики в canonical формате")
print("   • ❌ НЕТ суммирования в одну метрику accounts_payable")

print("\n📋 ОФИЦИАЛЬНЫЙ ОТЧЕТ:")
print("   • Accounts payable and other accrued liabilities: $2,601M")
print("   • Accounts payable to related parties: $146M (отдельная статья)")
print("   • Total current liabilities = $2,601M + $146M + другие статьи = $3,373M")

print("\n💡 ЛОГИКА МОДЕЛИРОВАНИЯ:")
print("   • AP моделируется через оборачиваемость (Accounts Payable Turnover = COGS / AP)")
print("   • Детализация отчетности (related parties vs regular AP) не нужна для моделирования")
print("   • Для удобства моделирования нужно ОБЪЕДИНИТЬ в одну метрику accounts_payable")
print("   • accounts_payable = accounts_payable_and_other_accrued_liabilities + accounts_payable_to_related_parties")
print("   • Ожидаемое значение: $2,601M + $146M = $2,747M")

print("\n✅ ВЫВОД:")
print("   • Текущее состояние: ДВЕ отдельные метрики (неправильно для моделирования)")
print("   • Нужно: ОДНА метрика accounts_payable = $2,747M (сумма обеих)")
print("   • accounts_payable_related_parties НЕ должна быть отдельной метрикой в canonical")
print("   • Она должна суммироваться в accounts_payable при препроцессинге")

mart.close()


In [ ]:
# === Пример: Где может понадобиться суммирование ===

display(Markdown("### 📋 Примеры статей, которые МОГУТ объединяться"))

# Примеры потенциального объединения
potential_merges = {
    'accounts_payable': {
        'description': 'ОБЪЕДИНЕНИЕ всех видов AP в одну метрику для моделирования через оборачиваемость',
        'source_metrics': [
            'accounts_payable_and_other_accrued_liabilities',  # $2,601M
            'accounts_payable_to_related_parties',  # $146M → accounts_payable_related_parties
        ],
        'target': 'accounts_payable',
        'current_status': '❌ НЕ объединяются (НУЖНО ИСПРАВИТЬ)',
        'when_needed': 'ВСЕГДА - AP моделируется через оборачиваемость, детализация не нужна',
        'expected_value': 2601.0 + 146.0,  # $2,747M
        'calculation': 'AP Turnover = COGS / accounts_payable',
    },
    'other_current_liabilities': {
        'description': 'Объединение различных прочих текущих обязательств',
        'source_metrics': [
            'accrued_liabilities',
            'taxes_payable',
            'interest_payable',
            'other_accrued_expenses',
        ],
        'target': 'other_current_liabilities',
        'current_status': '⚠️ Частично объединяются в preprocess.py',
        'when_needed': 'Когда нужно собрать все прочие обязательства в одну статью',
    },
    'other_non_current_liabilities': {
        'description': 'Объединение прочих долгосрочных обязательств',
        'source_metrics': [
            'deferred_credits',
            'other_long_term_liabilities',
            'provisions',
        ],
        'target': 'other_non_current_liabilities',
        'current_status': '⚠️ Частично объединяются в preprocess.py',
        'when_needed': 'Когда нужно собрать все прочие долгосрочные обязательства',
    },
}

print("\n" + "="*80)
print("📊 ПРИМЕРЫ ПОТЕНЦИАЛЬНОГО ОБЪЕДИНЕНИЯ")
print("="*80)

for target, info in potential_merges.items():
    print(f"\n🔍 {target.upper()}:")
    print(f"   Описание: {info['description']}")
    print(f"   Исходные метрики: {', '.join(info['source_metrics'])}")
    print(f"   Целевая метрика: {info['target']}")
    print(f"   Текущий статус: {info['current_status']}")
    print(f"   Когда нужно: {info['when_needed']}")
    if 'expected_value' in info:
        print(f"   Ожидаемое значение: ${info['expected_value']:,.0f}M")
    if 'calculation' in info:
        print(f"   Расчет моделирования: {info['calculation']}")

print("\n" + "="*80)
print("💡 ГДЕ ЭТО УЖЕ РЕАЛИЗОВАНО")
print("="*80)

print("\n📋 В preprocess.py (_process_calculated_metrics):")
print("   • other_current_assets = суммируются различные прочие активы")
print("   • other_current_liabilities = суммируются различные прочие обязательства")
print("   • other_non_current_assets = суммируются различные прочие долгосрочные активы")
print("   • other_non_current_liabilities = суммируются различные прочие долгосрочные обязательства")

print("\n📋 Логика суммирования:")
print("   • Ищет метрики по списку алиасов")
print("   • Суммирует все найденные значения")
print("   • Сохраняет как calculated метрику")

print("\n" + "="*80)
print("🔧 КАК ДОБАВИТЬ НОВОЕ ОБЪЕДИНЕНИЕ")
print("="*80)

print("\n1. 📄 В excel_loader.yaml добавить секцию 'combine_from' для accounts_payable:")
print("""
history:
  BS:
    required_metrics:
      accounts_payable:
        aliases:
          - accounts_payable_and_other_accrued_liabilities
        combine_from:  # НОВАЯ СЕКЦИЯ - метрики для суммирования
          - accounts_payable_and_other_accrued_liabilities  # $2,601M
          - accounts_payable_to_related_parties  # $146M (маппится на accounts_payable_related_parties)
        # Результат: accounts_payable = $2,601M + $146M = $2,747M
      
      # accounts_payable_related_parties НЕ должна быть в required_metrics
      # Она используется только как источник для accounts_payable
""")

print("\n2. 💻 В data_mart.py (normalize_to_canonical) добавить логику:")
print("""
# После создания canonical_df, перед заполнением значений:
for metric in required_metrics:
    metric_config = bs_metrics.get(metric, {})
    combine_from = metric_config.get('combine_from', [])
    
    if combine_from:
        # Суммируем все метрики из combine_from
        combined_value = 0.0
        for source_metric in combine_from:
            # Ищем source_metric в исходных данных
            source_row = df[df['metric'].str.lower() == source_metric.lower()]
            if not source_row.empty:
                for year in years:
                    year_str = str(year)
                    val = pd.to_numeric(source_row.iloc[0].get(year_str, 0), errors='coerce')
                    if pd.notna(val):
                        combined_value += float(val)
        
        # Записываем суммированное значение
        canonical_df.loc[canonical_df['metric'] == metric, year_str] = combined_value
""")

print("\n3. ✅ Результат:")
print("   • accounts_payable будет автоматически суммироваться из двух источников")
print("   • accounts_payable = $2,601M + $146M = $2,747M")
print("   • accounts_payable_related_parties НЕ будет в canonical формате")
print("   • Настраивается через YAML без изменения кода")
print("   • Работает при каждом вызове normalize_to_canonical")
print("   • AP Turnover = COGS / accounts_payable будет рассчитываться правильно")

print("\n" + "="*80)
print("💡 ЛОГИКА ОБЪЕДИНЕНИЯ СТАТЕЙ")
print("="*80)
print("\n📋 ПРИНЦИПЫ:")
print("   1. Детализация отчетности ≠ Детализация для моделирования")
print("   2. Если метрика моделируется через агрегированный показатель (turnover, ratio),")
print("      то детализация не нужна - объединяем в одну метрику")
print("   3. Примеры объединения:")
print("      • accounts_payable: моделируется через AP Turnover = COGS / AP")
print("        → Объединяем все виды AP в одну метрику")
print("      • accounts_receivable: моделируется через AR Turnover = Revenue / AR")
print("        → Объединяем все виды AR в одну метрику (если есть детализация)")
print("      • inventory: моделируется через Inventory Turnover = COGS / Inventory")
print("        → Объединяем все виды inventory в одну метрику")
print("\n   4. Когда НЕ объединять:")
print("      • Метрики моделируются отдельно (разные драйверы)")
print("      • Метрики имеют разную динамику и требуют отдельного прогноза")
print("      • Метрики используются в разных частях модели независимо")

print("\n📋 ДЛЯ accounts_payable:")
print("   ✅ ОБЪЕДИНЯЕМ: accounts_payable + accounts_payable_related_parties")
print("   ✅ Причина: Обе моделируются через один показатель - AP Turnover")
print("   ✅ Результат: accounts_payable = $2,747M (сумма обеих)")
print("   ✅ accounts_payable_related_parties НЕ должна быть в canonical формате")


## 🤖 ШАГ 11: Автоматический анализ и предложение объединения статей

**Цель**: Автоматически проанализировать все метрики и предложить варианты объединения

**Алгоритм**:
1. Проверить все canonical метрики - какие уже заполнены и однозначно не требуют объединения
2. Проверить незаполненные метрики - что осталось
3. Проанализировать RAW данные - какие метрики есть в исходных данных
4. Предложить варианты объединения на основе:
   - Логики моделирования (turnover, ratio)
   - Схожести названий
   - Структуры отчетности


In [ ]:
# === Автоматический анализ метрик и предложение объединения ===

display(Markdown("### 🤖 Автоматический анализ: Проверка заполненных и предложение объединения"))

mart = get_data_mart(ROOT, COMPANY)

# Загружаем данные
bs_raw = mart.get_history_balance_sheet(canonical=False)
bs_canonical = mart.get_history_balance_sheet(canonical=True)

# Загружаем конфигурацию
yaml_path = croot / 'configs' / 'excel_loader.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    loader_config = yaml.safe_load(f)

bs_metrics_config = loader_config.get('history', {}).get('BS', {}).get('required_metrics', {})

# Получаем список canonical метрик из БД
canonical_metrics_from_db = mart.get_canonical_metrics_from_db('BS')
if not canonical_metrics_from_db:
    # Fallback: используем из конфига
    canonical_metrics_from_db = list(bs_metrics_config.keys())

year_to_check = 2024
year_str = str(year_to_check)

print("\n" + "="*80)
print("📊 ШАГ 1: АНАЛИЗ ЗАПОЛНЕННЫХ МЕТРИК")
print("="*80)

# Анализируем каждую canonical метрику
filled_metrics = []
unfilled_metrics = []
metrics_needing_merge = []

for metric in canonical_metrics_from_db:
    # Проверяем значение в canonical формате
    canonical_val = None
    if not bs_canonical.empty and 'metric' in bs_canonical.columns:
        metric_row = bs_canonical[bs_canonical['metric'].str.lower() == metric.lower()]
        if not metric_row.empty and year_str in metric_row.columns:
            canonical_val = pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')
    
    # Проверяем, есть ли метрика в RAW данных
    raw_sources = []
    if not bs_raw.empty and 'metric' in bs_raw.columns:
        for raw_metric in bs_raw['metric'].unique():
            if pd.notna(raw_metric):
                raw_row = bs_raw[bs_raw['metric'] == raw_metric]
                if not raw_row.empty and year_str in raw_row.columns:
                    raw_val = pd.to_numeric(raw_row.iloc[0][year_str], errors='coerce')
                    if pd.notna(raw_val) and abs(raw_val) > 1e-6:
                        # Проверяем маппинг
                        metric_config = bs_metrics_config.get(metric, {})
                        aliases = metric_config.get('aliases', [])
                        if (raw_metric.lower() == metric.lower() or 
                            raw_metric.lower() in [a.lower() for a in aliases]):
                            raw_sources.append({
                                'metric': raw_metric,
                                'value': raw_val
                            })
    
    # Классифицируем метрику
    if pd.notna(canonical_val) and abs(canonical_val) > 1e-6:
        filled_metrics.append({
            'metric': metric,
            'value': canonical_val,
            'sources': raw_sources,
            'status': '✅ Заполнена',
            'needs_merge': len(raw_sources) > 1  # Если несколько источников - возможно нужно объединить
        })
    else:
        unfilled_metrics.append({
            'metric': metric,
            'sources': raw_sources,
            'status': '❌ Не заполнена' if len(raw_sources) == 0 else '⚠️ Есть источники, но не заполнена'
        })

print(f"\n📊 Статистика:")
print(f"   Всего canonical метрик: {len(canonical_metrics_from_db)}")
print(f"   ✅ Заполненных: {len(filled_metrics)}")
print(f"   ❌ Незаполненных: {len(unfilled_metrics)}")

# Показываем заполненные метрики с несколькими источниками (возможно нуждаются в объединении)
metrics_with_multiple_sources = [m for m in filled_metrics if m['needs_merge']]
if metrics_with_multiple_sources:
    print(f"\n⚠️ Метрики с несколькими источниками (возможно нужно объединить): {len(metrics_with_multiple_sources)}")
    for m in metrics_with_multiple_sources[:10]:  # Показываем первые 10
        print(f"   • {m['metric']}: {len(m['sources'])} источников")
        for src in m['sources']:
            print(f"      - {src['metric']}: ${src['value']:,.0f}")

mart.close()


In [ ]:
# === ШАГ 2: Анализ незаполненных метрик и поиск потенциальных источников ===

display(Markdown("### 🔍 ШАГ 2: Анализ незаполненных метрик"))

mart = get_data_mart(ROOT, COMPANY)
bs_raw = mart.get_history_balance_sheet(canonical=False)

year_to_check = 2024
year_str = str(year_to_check)

print("\n" + "="*80)
print("📊 АНАЛИЗ НЕЗАПОЛНЕННЫХ МЕТРИК")
print("="*80)

# Получаем все RAW метрики с ненулевыми значениями
raw_metrics_with_values = {}
if not bs_raw.empty and 'metric' in bs_raw.columns:
    for _, row in bs_raw.iterrows():
        raw_metric = row['metric']
        if pd.notna(raw_metric) and year_str in row.index:
            raw_val = pd.to_numeric(row[year_str], errors='coerce')
            if pd.notna(raw_val) and abs(raw_val) > 1e-6:
                raw_metrics_with_values[raw_metric] = raw_val

print(f"\n📊 RAW метрики с данными за {year_to_check}: {len(raw_metrics_with_values)}")

# Анализируем незаполненные canonical метрики
unfilled_analysis = []

for metric_info in unfilled_metrics:
    metric = metric_info['metric']
    
    # Ищем потенциальные источники по названию
    potential_sources = []
    
    # 1. Прямое совпадение
    if metric in raw_metrics_with_values:
        potential_sources.append({
            'metric': metric,
            'value': raw_metrics_with_values[metric],
            'match_type': 'прямое совпадение',
            'confidence': 'высокая'
        })
    
    # 2. Частичное совпадение
    metric_lower = metric.lower()
    for raw_metric, raw_val in raw_metrics_with_values.items():
        raw_metric_lower = raw_metric.lower()
        
        # Проверяем различные варианты совпадения
        if metric_lower in raw_metric_lower or raw_metric_lower in metric_lower:
            if raw_metric not in [s['metric'] for s in potential_sources]:
                potential_sources.append({
                    'metric': raw_metric,
                    'value': raw_val,
                    'match_type': 'частичное совпадение',
                    'confidence': 'средняя'
                })
    
    # 3. По ключевым словам
    keywords_map = {
        'payable': ['payable', 'ap', 'accounts_payable'],
        'receivable': ['receivable', 'ar', 'accounts_receivable'],
        'debt': ['debt', 'borrowings', 'loan'],
        'lease': ['lease', 'rou'],
        'tax': ['tax', 'taxes'],
        'accrued': ['accrued', 'accrual'],
    }
    
    for keyword, variants in keywords_map.items():
        if keyword in metric_lower:
            for raw_metric, raw_val in raw_metrics_with_values.items():
                raw_metric_lower = raw_metric.lower()
                if any(v in raw_metric_lower for v in variants):
                    if raw_metric not in [s['metric'] for s in potential_sources]:
                        potential_sources.append({
                            'metric': raw_metric,
                            'value': raw_val,
                            'match_type': f'ключевое слово: {keyword}',
                            'confidence': 'низкая'
                        })
    
    unfilled_analysis.append({
        'canonical_metric': metric,
        'potential_sources': potential_sources,
        'sources_count': len(potential_sources)
    })

# Сортируем по количеству потенциальных источников
unfilled_analysis.sort(key=lambda x: x['sources_count'], reverse=True)

print(f"\n📋 Незаполненные метрики с потенциальными источниками:")
for item in unfilled_analysis[:15]:  # Показываем первые 15
    if item['sources_count'] > 0:
        print(f"\n   🔍 {item['canonical_metric']}:")
        print(f"      Найдено потенциальных источников: {item['sources_count']}")
        for src in item['potential_sources'][:3]:  # Показываем первые 3
            print(f"      • {src['metric']}: ${src['value']:,.0f} ({src['match_type']}, уверенность: {src['confidence']})")

mart.close()


In [ ]:
# === ШАГ 3: Предложение объединения на основе логики моделирования ===

display(Markdown("### 💡 ШАГ 3: Предложение объединения метрик"))

mart = get_data_mart(ROOT, COMPANY)
bs_raw = mart.get_history_balance_sheet(canonical=False)
bs_canonical = mart.get_history_balance_sheet(canonical=True)

year_to_check = 2024
year_str = str(year_to_check)

print("\n" + "="*80)
print("💡 ПРЕДЛОЖЕНИЯ ПО ОБЪЕДИНЕНИЮ МЕТРИК")
print("="*80)

# Определяем метрики, которые моделируются через turnover/ratio и должны объединяться
# На основе практики финансового моделирования и типичных объединений статей баланса
turnover_metrics = {
    'accounts_payable': {
        'description': 'AP моделируется через AP Turnover = COGS / AP',
        'keywords': ['payable', 'ap', 'accounts_payable'],
        'related_keywords': ['related_parties', 'related', 'accrued', 'trade'],
        'calculation': 'COGS / accounts_payable',
        'common_combinations': [
            ['accounts_payable', 'accounts_payable_to_related_parties'],
            ['accounts_payable', 'trade_payables', 'accounts_payable_to_related_parties'],
            ['accounts_payable_and_other_accrued_liabilities', 'accounts_payable_to_related_parties'],
        ],
        'rationale': 'Все виды кредиторской задолженности объединяются для расчета оборачиваемости AP',
    },
    'accounts_receivable': {
        'description': 'AR моделируется через AR Turnover = Revenue / AR',
        'keywords': ['receivable', 'ar', 'accounts_receivable', 'trade_receivable'],
        'related_keywords': ['related_parties', 'related', 'trade', 'other'],
        'calculation': 'Revenue / accounts_receivable',
        'common_combinations': [
            ['accounts_receivable', 'receivables_from_related_parties'],
            ['trade_receivables', 'receivables_from_related_parties', 'other_receivables'],
            ['accounts_receivable', 'receivables_related_parties'],
        ],
        'rationale': 'Все виды дебиторской задолженности объединяются для расчета оборачиваемости AR',
    },
    'inventory': {
        'description': 'Inventory моделируется через Inventory Turnover = COGS / Inventory',
        'keywords': ['inventory', 'inventories'],
        'related_keywords': ['raw_materials', 'work_in_progress', 'finished_goods', 'goods'],
        'calculation': 'COGS / inventory',
        'common_combinations': [
            ['raw_materials', 'work_in_progress', 'finished_goods'],
            ['inventory_raw_materials', 'inventory_wip', 'inventory_finished'],
        ],
        'rationale': 'Все виды запасов объединяются для расчета оборачиваемости Inventory',
    },
}

# Дополнительные метрики, которые часто объединяются (не turnover, но агрегируются)
aggregation_metrics = {
    'other_current_liabilities': {
        'description': 'Прочие текущие обязательства - агрегируются различные начисленные обязательства',
        'keywords': ['accrued', 'other_current', 'other_liabilities'],
        'related_keywords': ['accrued_liabilities', 'accrued_expenses', 'taxes_payable', 'interest_payable', 'other_accrued'],
        'common_combinations': [
            ['accrued_liabilities', 'accrued_expenses', 'other_accrued_liabilities'],
            ['taxes_payable', 'interest_payable', 'accrued_liabilities'],
            ['accrued_liabilities', 'other_current_liabilities'],
        ],
        'rationale': 'Различные начисленные обязательства объединяются в прочие текущие обязательства',
    },
    'other_non_current_liabilities': {
        'description': 'Прочие долгосрочные обязательства - агрегируются различные долгосрочные обязательства',
        'keywords': ['other_non_current', 'other_long_term', 'deferred'],
        'related_keywords': ['deferred_credits', 'deferred_liabilities', 'provisions', 'other_long_term'],
        'common_combinations': [
            ['deferred_credits', 'other_non_current_liabilities'],
            ['deferred_liabilities', 'provisions', 'other_long_term_liabilities'],
        ],
        'rationale': 'Различные долгосрочные обязательства объединяются в прочие долгосрочные обязательства',
    },
    'other_current_assets': {
        'description': 'Прочие текущие активы - агрегируются различные текущие активы',
        'keywords': ['other_current', 'prepaid', 'other_assets'],
        'related_keywords': ['prepaid_expenses', 'other_current_assets', 'short_term_investments'],
        'common_combinations': [
            ['prepaid_expenses', 'other_current_assets'],
            ['short_term_investments', 'other_current_assets'],
        ],
        'rationale': 'Различные текущие активы объединяются в прочие текущие активы',
    },
    'other_non_current_assets': {
        'description': 'Прочие долгосрочные активы - агрегируются различные долгосрочные активы',
        'keywords': ['other_non_current', 'other_long_term', 'investments'],
        'related_keywords': ['long_term_investments', 'other_non_current_assets', 'investments_associates'],
        'common_combinations': [
            ['long_term_investments', 'investments_in_associates', 'other_non_current_assets'],
        ],
        'rationale': 'Различные долгосрочные активы объединяются в прочие долгосрочные активы',
    },
}

# Получаем все RAW метрики
raw_metrics_dict = {}
if not bs_raw.empty and 'metric' in bs_raw.columns:
    for _, row in bs_raw.iterrows():
        raw_metric = row['metric']
        if pd.notna(raw_metric) and year_str in row.index:
            raw_val = pd.to_numeric(row[year_str], errors='coerce')
            if pd.notna(raw_val):
                raw_metrics_dict[raw_metric] = raw_val

# Объединяем все метрики для анализа
all_metrics_to_check = {**turnover_metrics, **aggregation_metrics}

# Анализируем каждую метрику
merge_suggestions = []

for canonical_metric, config in all_metrics_to_check.items():
    # Проверяем текущее значение в canonical
    canonical_val = None
    if not bs_canonical.empty and 'metric' in bs_canonical.columns:
        metric_row = bs_canonical[bs_canonical['metric'].str.lower() == canonical_metric.lower()]
        if not metric_row.empty and year_str in metric_row.columns:
            canonical_val = pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')
    
    # Ищем все потенциальные источники
    potential_sources = []
    
    # Сначала проверяем типичные комбинации из практики
    common_combos = config.get('common_combinations', [])
    found_in_common_combos = set()
    
    for combo in common_combos:
        combo_found = []
        for combo_metric in combo:
            combo_metric_lower = combo_metric.lower()
            # Ищем в RAW данных
            for raw_metric, raw_val in raw_metrics_dict.items():
                raw_metric_lower = raw_metric.lower()
                if (combo_metric_lower == raw_metric_lower or 
                    combo_metric_lower in raw_metric_lower or 
                    raw_metric_lower in combo_metric_lower):
                    if raw_metric not in found_in_common_combos:
                        combo_found.append({
                            'metric': raw_metric,
                            'value': raw_val,
                            'match_reason': f'типичная комбинация: {combo_metric}',
                            'confidence': 'высокая'
                        })
                        found_in_common_combos.add(raw_metric)
        
        if len(combo_found) > 1:  # Если нашли несколько из комбинации
            potential_sources.extend(combo_found)
    
    # Затем ищем по ключевым словам (если еще не нашли)
    for raw_metric, raw_val in raw_metrics_dict.items():
        if raw_metric in found_in_common_combos:
            continue  # Уже добавлено через типичные комбинации
            
        raw_metric_lower = raw_metric.lower()
        
        # Проверяем совпадение по ключевым словам
        matches_main = any(kw in raw_metric_lower for kw in config['keywords'])
        matches_related = any(kw in raw_metric_lower for kw in config['related_keywords']) if config.get('related_keywords') else False
        
        if matches_main or (matches_main and matches_related):
            # Проверяем, не является ли это уже основной метрикой
            if raw_metric_lower != canonical_metric.lower():
                potential_sources.append({
                    'metric': raw_metric,
                    'value': raw_val,
                    'match_reason': 'ключевые слова' if matches_main else 'связанные ключевые слова',
                    'confidence': 'средняя'
                })
    
    # Если есть потенциальные источники для объединения
    if potential_sources:
        # Проверяем, не маппятся ли они уже на другие canonical метрики
        from engine.database.data_mart import METRIC_NAME_MAPPING
        bs_mapping = METRIC_NAME_MAPPING.get('BS', {})
        
        sources_to_merge = []
        for src in potential_sources:
            src_metric_lower = src['metric'].lower()
            # Проверяем, не маппится ли уже на другую canonical метрику
            mapped_to_other = False
            for alias, canonical in bs_mapping.items():
                if alias.lower() == src_metric_lower and canonical.lower() != canonical_metric.lower():
                    mapped_to_other = True
                    break
            
            if not mapped_to_other:
                sources_to_merge.append(src)
        
        if sources_to_merge:
            # Рассчитываем ожидаемое значение
            current_value = canonical_val if pd.notna(canonical_val) else 0.0
            sum_of_sources = sum(s['value'] for s in sources_to_merge)
            expected_value = current_value + sum_of_sources
            
            merge_suggestions.append({
                'canonical_metric': canonical_metric,
                'description': config['description'],
                'calculation': config['calculation'],
                'current_value': current_value,
                'sources_to_merge': sources_to_merge,
                'expected_value': expected_value,
                'needs_action': abs(current_value - expected_value) > 1e-6 if pd.notna(canonical_val) else True
            })

print(f"\n📊 Найдено предложений по объединению: {len(merge_suggestions)}")

for suggestion in merge_suggestions:
    print(f"\n🔍 {suggestion['canonical_metric'].upper()}:")
    print(f"   Описание: {suggestion['description']}")
    if 'calculation' in suggestion:
        print(f"   Расчет: {suggestion['calculation']}")
    if 'rationale' in suggestion:
        print(f"   Обоснование: {suggestion['rationale']}")
    print(f"   Текущее значение: ${suggestion['current_value']:,.0f}")
    print(f"   Источники для объединения ({len(suggestion['sources_to_merge'])}):")
    for src in suggestion['sources_to_merge']:
        confidence = src.get('confidence', 'средняя')
        match_reason = src.get('match_reason', 'не указано')
        print(f"      • {src['metric']}: ${src['value']:,.0f} ({match_reason}, уверенность: {confidence})")
    print(f"   Ожидаемое значение после объединения: ${suggestion['expected_value']:,.0f}")
    if suggestion['needs_action']:
        print(f"   ⚠️ ТРЕБУЕТСЯ ДЕЙСТВИЕ: Объединить источники в {suggestion['canonical_metric']}")
    else:
        print(f"   ✅ Уже объединено правильно")

mart.close()


In [ ]:
# === ШАГ 4: Генерация YAML конфигурации для объединения ===

display(Markdown("### 📝 ШАГ 4: Генерация конфигурации для excel_loader.yaml"))

print("\n" + "="*80)
print("📝 ПРЕДЛОЖЕНИЯ ДЛЯ excel_loader.yaml")
print("="*80)

print("\n💡 ИСТОЧНИКИ ЗНАНИЙ ОБ ОБЪЕДИНЕНИИ:")
print("   • Практика финансового моделирования (3-statement models)")
print("   • Типичные объединения статей баланса в бухгалтерском учете")
print("   • Логика моделирования через turnover/ratio метрики")
print("   • Агрегация статей для упрощения анализа")

# Используем результаты из предыдущих шагов
if 'merge_suggestions' in locals() and merge_suggestions:
    print("\n💡 Добавьте следующую конфигурацию в excel_loader.yaml:")
    print("\n```yaml")
    print("history:")
    print("  BS:")
    print("    required_metrics:")
    
    for suggestion in merge_suggestions:
        if suggestion['needs_action']:
            canonical = suggestion['canonical_metric']
            sources = [s['metric'] for s in suggestion['sources_to_merge']]
            
            print(f"\n      {canonical}:")
            print(f"        required_for:")
            print(f"          - custom")
            print(f"        aliases:")
            # Проверяем текущие алиасы из конфига
            current_config = bs_metrics_config.get(canonical, {})
            current_aliases = current_config.get('aliases', [])
            if current_aliases:
                for alias in current_aliases:
                    print(f"          - {alias}")
            else:
                # Предлагаем первый источник как алиас
                if sources:
                    print(f"          - {sources[0]}")
            
            print(f"        combine_from:  # НОВАЯ СЕКЦИЯ - метрики для суммирования")
            for src in sources:
                print(f"          - {src}")
            print(f"        # Ожидаемое значение: ${suggestion['expected_value']:,.0f}")
    
    print("```")
    
    print("\n📋 ИНСТРУКЦИЯ:")
    print("   1. Откройте файл: companies/us_steel/configs/excel_loader.yaml")
    print("   2. Найдите секцию history.BS.required_metrics")
    print("   3. Для каждой метрики выше добавьте секцию combine_from")
    print("   4. Сохраните файл")
    print("   5. Обновите код normalize_to_canonical в data_mart.py для поддержки combine_from")
    
else:
    print("\n⚠️ Нет предложений по объединению (возможно, все уже правильно настроено)")

print("\n" + "="*80)
print("✅ АНАЛИЗ ЗАВЕРШЕН")
print("="*80)
print("\n📋 СЛЕДУЮЩИЕ ШАГИ:")
print("   1. Проверьте предложения по объединению выше")
print("   2. Добавьте combine_from в excel_loader.yaml для нужных метрик")
print("   3. Реализуйте логику суммирования в normalize_to_canonical (data_mart.py)")
print("   4. Перезапустите препроцессинг и проверьте результаты")
